# **<center>1. Imports</center>**

In [1]:
# ============================================================
# 1. Imports and Environment Information
# ============================================================

# ----------------------------
# Standard libraries
# ----------------------------
import os
import sys
import re
import copy
import time
import random
import shutil
import math
import platform
import importlib.metadata

from pathlib import Path
from collections import Counter, defaultdict


# ----------------------------
# Data handling
# ----------------------------
import pandas as pd
import numpy as np
import scipy
from scipy.stats import skew
import joblib


# ----------------------------
# Visualization and image handling
# ----------------------------
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import PIL
from PIL import Image
import pillow_heif


# ----------------------------
# Scikit-learn
# ----------------------------
import sklearn

from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# ----------------------------
# PyTorch and TorchVision
# ----------------------------
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torch.utils.data import (
    DataLoader,
    Subset,
    random_split,
    ConcatDataset,
)

from torchvision import (
    datasets,
    models,
    transforms,
)

import torchvision.transforms.v2 as v2


# ----------------------------
# Enable HEIF / HEIC support
# ----------------------------
pillow_heif.register_heif_opener()


# ============================================================
# Environment logging
# ============================================================

print("=== Environment Info ===")

# System information
print(f"Platform: {platform.platform()}")
print(f"System: {platform.system()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor() or 'Not reported'}")

# Software versions
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Pillow: {PIL.__version__}")
print(f"Joblib: {joblib.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")

try:
    print(f"pillow-heif: {importlib.metadata.version('pillow-heif')}")
except importlib.metadata.PackageNotFoundError:
    print("pillow-heif: Not installed")

# Apple Metal Performance Shaders
if hasattr(torch.backends, "mps"):
    print(f"MPS built: {torch.backends.mps.is_built()}")
    print(f"MPS available: {torch.backends.mps.is_available()}")
else:
    print("MPS built: Not supported by this PyTorch build")
    print("MPS available: No")

=== Environment Info ===
Platform: macOS-14.5-arm64-arm-64bit
System: Darwin
Machine: arm64
Processor: arm
Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
PyTorch: 2.5.1
TorchVision: 0.20.1
NumPy: 1.26.4
Pandas: 2.2.2
SciPy: 1.13.1
Scikit-learn: 1.5.1
Pillow: 10.4.0
Joblib: 1.4.2
Matplotlib: 3.9.2
Seaborn: 0.13.2
pillow-heif: 0.21.0
MPS built: True
MPS available: True


In [2]:
# ============================================================
# Project / Dataset Paths
# ============================================================

from dotenv import load_dotenv

# Permanent private configuration
CONFIG_FILE = Path.home() / ".wadidegla.env"

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        "Local WadiDegla configuration file was not found."
    )

load_dotenv(CONFIG_FILE)


DATA_DIR = Path(os.environ["WADIDEGLA_DATA_DIR"])
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Validation"

WORKSPACE_DIR = Path(os.environ["WADIDEGLA_WORKSPACE_DIR"])

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "The configured WadiDegla dataset directory does not exist."
    )

if not WORKSPACE_DIR.exists():
    raise FileNotFoundError(
        "The configured WadiDegla experimental workspace directory does not exist."
    )

print("WadiDegla dataset configuration loaded successfully.")
print("WadiDegla experimental workspace configuration loaded successfully.")

WadiDegla dataset configuration loaded successfully.
WadiDegla experimental workspace configuration loaded successfully.


# **<center>2. Reproducibility Setup for Mac M1/MPS</center>**

In [3]:
def set_seed(seed: int, deterministic: bool = True):
    """
    Initialize all RNG seeds for reproducibility on Apple Silicon M1.
    Optimized for MPS backend.

    Returns:
        start_rng_state (dict): RNG states captured immediately after seeding.
        generator (torch.Generator): Torch generator initialized with the same seed.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True)

    generator = torch.Generator()
    generator.manual_seed(seed)

    rng_state = {
        'seed': seed,
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_torch_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
        'deterministic': deterministic,
        'generator_state': generator.get_state()
    }
    print(f"✅ Random seed set to {seed} (MPS available: {torch.backends.mps.is_available()})")
    return rng_state, generator

In [4]:
def get_rng_state(seed: int, generator: torch.Generator, deterministic: bool = True):
    """
    Capture the CURRENT RNG states for Python, NumPy, and PyTorch (CPU + MPS if available).

    Returns:
        (dict, torch.Generator): RNG state dict and the generator.
    """
    rng_state = {
        'seed': seed,
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_torch_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
        'deterministic': deterministic,
        'generator_state': generator.get_state()
    }
    return rng_state, generator

In [5]:
def manage_rng_state(path: str, mode: str, seed: int = None, generator: torch.Generator = None, deterministic: bool = True):
    """
    Save or load RNG state for Python, NumPy, and PyTorch (CPU + MPS).

    Returns:
        (dict, torch.Generator): RNG state dict and the generator.
    """
    if mode == 'load':
        rng_state = joblib.load(path)

        random.setstate(rng_state['rng_python'])
        np.random.set_state(rng_state['rng_numpy'])
        torch.set_rng_state(rng_state['rng_torch'])
        if torch.backends.mps.is_available() and rng_state.get('rng_torch_mps') is not None:
            torch.mps.set_rng_state(rng_state['rng_torch_mps'])
        torch.use_deterministic_algorithms(rng_state.get('deterministic', False))

        generator = torch.Generator()
        if 'generator_state' in rng_state:
            generator.set_state(rng_state['generator_state'])
        elif seed is not None:
            generator.manual_seed(seed)

        print(f"✅ RNG state loaded from {path}")
        return rng_state, generator

    elif mode == 'save':
        if generator is None:
            generator = torch.Generator().manual_seed(seed)
        rng_state, generator = get_rng_state(seed, generator, deterministic)
        joblib.dump(rng_state, path)
        print(f"✅ RNG state saved to {path}")
        return rng_state, generator

    else:
        raise ValueError("mode must be either 'save' or 'load'")

In [6]:
def setup_reproducibility(start_fresh: bool = False,
                          seed: int = None,
                          deterministic: bool = True,
                          continue_run: bool = False,
                          path: str = None,
                          mode: str = None,
                          generator: torch.Generator = None):
    """
    Wrapper for reproducibility setup:
      - Optionally set RNG seeds.
      - Optionally save/load RNG states.

    Returns:
        (dict, torch.Generator)
    """
    if start_fresh:
        if seed is None:
            raise ValueError("`seed` must be provided when start_fresh=True.")
        return set_seed(seed=seed, deterministic=deterministic)

    if continue_run:
        if path is None or mode is None:
            raise ValueError("Both `path` and `mode` must be provided when continue_run=True.")
        return manage_rng_state(path=path, mode=mode, seed=seed, generator=generator, deterministic=deterministic)

**Start fresh with a new seed**

In [7]:
seeds = [42, 1234, 2025]

In [8]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


**Save current RNG state**

In [10]:
state, generator = setup_reproducibility(continue_run=True, path="mobilenetv2/rng_state.pkl", mode="save", seed=seeds[0], generator=generator)

✅ RNG state saved to mobilenetv2/rng_state.pkl


**Load previously saved RNG state**

In [11]:
state, generator = setup_reproducibility(continue_run=True, path="mobilenetv2/rng_state.pkl", mode="load")

✅ RNG state loaded from mobilenetv2/rng_state.pkl


# <center>**Input Pipeline Setup (Dataset Loading & Transformations)**</center>

## **1. Experiment Configuration**

In [12]:
BATCH_SIZE = 32
NUM_CLASSES = sum(1 for d in (DATA_DIR/'Train').iterdir() if d.is_dir())
NUM_CLASSES

33

## **2. Define Data Augmentation & Preprocessing Pipeline**

**Resize then Augment**
- **When working with plant images of varying sizes, the recommended order is: resize first, then augment.**


1. **Consistency:** Resizing first ensures all images share the same dimensions, leading to consistent transformation behavior during augmentation.

2. **Computational Efficiency:** Augmentation is faster on smaller, uniform images, while large images use more memory and slow training.
  
3. **Avoids Distortion Cascading:** Resizing first provides a clean base, preventing amplified artifacts (blur, aliasing, edge blanks) that occur when resizing after augmentation.
  

In [13]:
def build_train_transform(
    crop_type="random",
    scale=(0.2, 1.0),
    crop_size=224,
    ratio=(0.75, 1.33),
    color_jitter=True,
    jitter_prob=0.8,
    always_jitter=False,
    hflip_prob=0.5,
    vflip_prob=0.2,
    rotate_prob=0.5,
    rotation_degrees=(-90, 90),
    brightness=(0.7, 1.3),
    contrast=(0.85, 1.3),
    saturation=(0.7, 1.3),
    hue=(-0.0278, 0.0278),
    mean=None,
    std=None,
    normalize=True
):
    """
    Build a training transform pipeline with customizable augmentations.
    ...
    """
    mean = mean or [0.485, 0.456, 0.406]
    std = std or [0.229, 0.224, 0.225]
    
    # Stage 1: Input Preparation & Spatial Augmentations
    transforms_list = [v2.ToImage()]

    if crop_type == "random":
        transforms_list.append(
            v2.RandomResizedCrop(
                size=(crop_size, crop_size),
                scale=scale,
                ratio=ratio,
                antialias=True
            )
        )
    elif crop_type == "center":
        transforms_list.extend([
            v2.Resize(256),
            v2.CenterCrop(crop_size)
        ])
    else:
        raise ValueError(f"Unknown crop_type: {crop_type}. Use 'random' or 'center'")

    transforms_list.extend([
        v2.RandomHorizontalFlip(p=hflip_prob),
        v2.RandomVerticalFlip(p=vflip_prob),
        v2.RandomApply([
            v2.RandomRotation(degrees=rotation_degrees, interpolation=v2.InterpolationMode.BILINEAR)
        ], p=rotate_prob)
    ])

    # Stage 2: Color Operations
    transforms_list.append(v2.ToDtype(torch.float32, scale=True)) # Convert to [0,1] float32

    if color_jitter:
        jitter = v2.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue
        )
        transforms_list.append(jitter if always_jitter else v2.RandomApply([jitter], p=jitter_prob)) # Apply color jitter jitter_prob of the time, identity (1 - jitter_prob)

    # Stage 3: Normalization (CRUCIAL for training)
    if normalize:
        transforms_list.append(v2.Normalize(mean=mean, std=std))

    return v2.Compose(transforms_list)

In [14]:
default_train_transform = build_train_transform(crop_type="random", scale=(0.2, 1.0))
# crop_50_train_transform = build_train_transform(crop_type="random", scale=(0.5, 1.0))
# crop_80_train_transform = build_train_transform(crop_type="random", scale=(0.8, 1.0))
# center_crop_train_transform = build_train_transform(crop_type="center")
# hard_train_transform = build_train_transform(crop_type="random", scale=(0.08, 1.0), always_jitter=True)

In [15]:
default_train_transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

**In validation, it is best to first resize the image so that its shorter side is 256 pixels—whether by upscaling or downscaling—and then apply a center crop of 224×224 to obtain a well-centered, standardized region.**

In [16]:
# Validation transforms (no augmentation)
val_transform = v2.Compose([
    v2.ToImage(),
    # Resize the image so its shorter side is 256 pixels, preserving aspect ratio (i.e., the proportion between width and height stays the same)
    v2.Resize(256),
    # Extract a 224x224 patch from the center of the resized image
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## **3. Dataset Construction**

### **3.1 File Extension Validation**

In [17]:
def count_extensions(root_dir):
    extension_counter = defaultdict(int)
    c = 0
    for foldername, subfolders, filenames in os.walk(root_dir):
      
            
        for filename in filenames:
            # Extract extension and convert to lowercase
            ext = os.path.splitext(filename)[1].lower()
            if ext == '':
                continue
            # Count extension (even if empty)
            extension_counter[ext] += 1
    
    return dict(extension_counter)

# Usage
extension_counts = count_extensions(DATA_DIR)

print("Extension counts:")
for ext, count in extension_counts.items():
    print(f"{ext}: {count}")
print(f'Total: {sum(extension_counts.values())}')    

Extension counts:
.jpg: 21042
.jpeg: 1522
Total: 22564


### **3.2 Dataset creation**

In [18]:
# Create Dataset objects for training and validation sets

train_dataset = datasets.ImageFolder(
    root=DATA_DIR / "Train"  # Path to the training directory
)


val_dataset = datasets.ImageFolder(
    root=DATA_DIR / "Validation"  # Path to the validation directory
)

print(f"Training dataset size: {len(train_dataset)} images")
print(f"Validation dataset size: {len(val_dataset)} images")

Training dataset size: 17364 images
Validation dataset size: 5200 images


In [60]:
print(f'Number of model classes: {len(train_dataset.classes)}')
print(f'Number of images: {len(train_dataset.targets)+len(val_dataset.targets)}')

Number of model classes: 33
Number of images: 22564


In [20]:
train_dataset.transform =default_train_transform  # Apply the aggressive augmentation pipeline
val_dataset.transform   = val_transform    # Apply the deterministic preprocessing pipeline 

In [21]:
train_dataset.transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

In [22]:
val_dataset.transform  

Compose(
      ToImage()
      Resize(size=[256], interpolation=InterpolationMode.BILINEAR, antialias=True)
      CenterCrop(size=(224, 224))
      ToDtype(scale=True)
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

## **4. DataLoader Configuration**

**This is where we wrap the datasets into DataLoaders for batching, shuffling, and parallel loading.**

In [23]:
# Detect if CUDA is available; on M1 we expect only MPS/CPU
use_cuda = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # shuffle ensures different order each epoch
    num_workers=4,            # safer on macOS, avoids multiprocessing issues
    pin_memory=True,     
    generator=generator       # ensures reproducible shuffling
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,            # no need to shuffle validation
    num_workers=4,            # same note for macOS
    pin_memory=True,
    generator=generator
)

In [24]:
# Verify the loaders
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 543
Validation batches: 163


# <center>**Utility Functions for Model Training & Evaluation**</center>

## **1. Compute Class Weights**

In [29]:
def compute_class_weights(dataset, method="inverse_frequency", normalize=False):
    """
    Compute class weights based on the chosen method, with optional normalization.

    Args:
        dataset: A dataset object with a 'targets' attribute containing class labels.
        method (str): Method for computing class weights. Options:
            - "inverse_frequency": total_samples / count
            - "balanced": total_samples / (count * num_classes)
            - "sqrt_inverse": 1 / np.sqrt(count)
            - "scaled_sqrt_inverse": (1 / np.sqrt(count)) * 100
        normalize (bool): If True, rescale weights so that their mean = 1.

    Returns:
        torch.Tensor: Class weights tensor on the appropriate device.
    """
    # Compute class frequencies
    class_counts = Counter(dataset.targets)
    total_samples = sum(class_counts.values())
    num_classes = len(class_counts)

    # Select weight calculation method
    if method == "inverse_frequency":
        class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
    elif method == "balanced":
        class_weights = {cls: total_samples / (count * num_classes) for cls, count in class_counts.items()}
    elif method == "sqrt_inverse":
        class_weights = {cls: 1 / np.sqrt(count) for cls, count in class_counts.items()}
    elif method == "scaled_sqrt_inverse":
        class_weights = {cls: (1 / np.sqrt(count)) * 100 for cls, count in class_counts.items()}
    else:
        raise ValueError("Invalid method. Choose from 'inverse_frequency', 'balanced', 'sqrt_inverse', 'scaled_sqrt_inverse'.")

    # Convert to tensor
    class_weights_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float32)

    #  Normalize weights to have an average of 1.0 → divide by mean 
    if normalize:
        class_weights_tensor = class_weights_tensor / class_weights_tensor.mean()

    # Move weights to MPS (Apple GPU) if available
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    return class_weights_tensor.to(device)

In [30]:
class_index_count = {train_dataset.classes[i]:sorted(Counter(train_dataset.targets).items())[i] for i in range(len(train_dataset.classes))}
print(class_index_count)

{'Anabasis articulata (Forssk.) Moq.': (0, 299), 'Anabasis setifera Moq.': (1, 865), 'Atriplex halimus L.': (2, 957), 'Calotropis procera (Aiton) W.T.Aiton': (3, 266), 'Capparis spinosa L.': (4, 515), 'Cebatha pendula (J.R.Forst. & G.Forst.) Kuntze': (5, 368), 'Cenchrus divisus (J.F.Gmel.) Verloove, Govaerts & Buttler': (6, 205), 'Deverra tortuosa (Desf.) DC.': (7, 156), 'Deverra triradiata Hochst. ex Boiss.': (8, 148), 'Diplotaxis harra (Forssk.) Boiss.': (9, 415), 'Echinops glaberrimus DC.': (10, 734), 'Ephedra alata Decne.': (11, 860), 'Farsetia aegyptia Turra': (12, 419), 'Gymnocarpos decander Forssk.': (13, 571), 'Haloxylon salicornicum (Moq.) Bunge ex Boiss.': (14, 672), 'Heliotropium arbainense Fresen.': (15, 674), 'Iphiona mucronata (Forssk.) Asch. & Schweinf.': (16, 110), 'Limonium pruinosum (L.) Chaz.': (17, 307), 'Lycium shawii Roem. & Schult.': (18, 1073), 'Nitraria retusa (Forssk.) Asch.': (19, 704), 'Ochradenus baccatus Delile': (20, 1396), 'Peganum harmala L.': (21, 85),

In [31]:
inverse_frequency_class_weights = compute_class_weights(train_dataset, method="inverse_frequency")
normalized_inverse_frequency_class_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=True)

In [32]:
print(f"Inverse frequency class weights: {[round(v, 2) for v in inverse_frequency_class_weights.cpu().numpy().tolist()]}")
print(f"Inverse frequency class weights: {round(inverse_frequency_class_weights.mean().item(), 2)}")


Inverse frequency class weights: [58.07, 20.07, 18.14, 65.28, 33.72, 47.18, 84.7, 111.31, 117.32, 41.84, 23.66, 20.19, 41.44, 30.41, 25.84, 25.76, 157.85, 56.56, 16.18, 24.66, 12.44, 204.28, 51.07, 42.66, 44.41, 40.76, 16.17, 53.76, 38.08, 19.78, 25.99, 85.96, 43.41]
Inverse frequency class weights: 51.48


In [33]:
print(f"Normalized Inverse frequency class weights: {[round(v, 2) for v in normalized_inverse_frequency_class_weights.cpu().numpy().tolist()]}")
print(f"Normalized Inverse frequency class weights: {round(normalized_inverse_frequency_class_weights.mean().item(), 2)}")

Normalized Inverse frequency class weights: [1.13, 0.39, 0.35, 1.27, 0.65, 0.92, 1.65, 2.16, 2.28, 0.81, 0.46, 0.39, 0.8, 0.59, 0.5, 0.5, 3.07, 1.1, 0.31, 0.48, 0.24, 3.97, 0.99, 0.83, 0.86, 0.79, 0.31, 1.04, 0.74, 0.38, 0.5, 1.67, 0.84]
Normalized Inverse frequency class weights: 1.0


In [34]:
def compute_inverse_metric_tensor(
    metric_dict, 
    class_to_idx, 
    normalize=True, 
    power=1.0, 
    eps=1e-8,
    clip = False,
    clip_min=0.5, 
    clip_max=3.0
):
    """
    Compute inverse metric-based class weights and return as a PyTorch tensor.

    Parameters
    ----------
    metric_dict : dict
        {class_name: metric_value}, e.g. recall, precision, or F1-score.
    class_to_idx : dict
        Mapping of {class_name: class_index} from the dataset.
    normalize : bool, optional
        Normalize class_weights_tensor to mean 1. Default True.
    power : float, optional
        Exponent controlling the scaling. Default 1.0.
    eps : float, optional
        Small constant to avoid division by zero.
    clip_min, clip_max : float
        Minimum and maximum clipping values for stability.

    Returns
    -------
    torch.Tensor
        Tensor of class weights ordered by class index, dtype=torch.float32.
    """
    # Compute inverse weights in the correct class order
    inv_weights = []
    for cls, idx in class_to_idx.items():
        metric = metric_dict.get(cls)
        inv_w = (1.0 / (metric + eps)) ** power
        inv_weights.append(inv_w)

    # Convert to tensor
    class_weights_tensor = torch.tensor(inv_weights, dtype=torch.float32)

    # Clip weights to avoid extremes
    if clip:
        class_weights_tensor = torch.clamp(class_weights_tensor, min=clip_min, max=clip_max)

    # Normalize (mean = 1)
    if normalize:
        class_weights_tensor = class_weights_tensor / class_weights_tensor.mean()

    # Move to device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    return class_weights_tensor.to(device)

## **2. Candidate Models Configuration**

### **1. mobilenet_v2**

In [35]:
def configure_mobilenetv2_training(
    model: nn.Module,
    num_classes: int,
    stage_a: bool = False,
    intermediate: bool = False,
    intermediate_blocks: list[int] | None = None,
    final_stage: bool = False,
    prev_unfrozen: int | None = None,
):
    """
    Configure MobileNetV2 model for staged training (Stage A, Intermediate, Final).

    Args:
        model (nn.Module): MobileNetV2 model instance (fresh or from previous stage).
        num_classes (int): Number of output classes for the dataset.
        stage_a (bool): If True, reinitialize classifier head and freeze backbone.
        intermediate (bool): If True, progressively unfreeze backbone blocks.
        intermediate_blocks (list[int] | None): [prev_unfrozen, new_unfrozen].
            Example: [1, 3] → model had 1 block unfrozen before, now unfreeze 3.
        final_stage (bool): If True, unfreeze the entire backbone.
        prev_unfrozen (int | None): Number of previously unfrozen blocks (for final stage).

    Returns:
        nn.Module: Model with updated training configuration.

    Prints:
        - Trainable layers count and % of total layers
        - Trainable parameters count and % of total parameters
        - Which backbone blocks are trainable
    """

    # ---- Stage A: train only classifier head ----
    if stage_a:
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

        # Kaiming normal init
        nn.init.kaiming_normal_(model.classifier[1].weight, nonlinearity="relu")
        if model.classifier[1].bias is not None:
            nn.init.zeros_(model.classifier[1].bias)

        # Freeze everything
        for param in model.parameters():
            param.requires_grad = False
        
        # Unfreeze the entire classifier
        for param in model.classifier.parameters():
            param.requires_grad = True  
        print("🟢 Stage A → Classifier head is trainable, backbone frozen.")
            

    # ---- Intermediate stage: progressively unfreeze ----
    if intermediate:
        assert intermediate_blocks is not None and len(intermediate_blocks) == 2, \
            "intermediate_blocks must be [prev_unfrozen, new_unfrozen]."

        prev_unfrozen, new_unfrozen = intermediate_blocks
        assert new_unfrozen >= prev_unfrozen, \
            "New unfrozen count must be >= previous unfrozen count."

   
    
        # Unfreeze last `new_unfrozen` blocks
        total_blocks = len(model.features)
        prev_frozen_blocks = total_blocks - prev_unfrozen
        new_frozen_blocks = total_blocks - new_unfrozen

        unfrozen_indices = []
        
        for i in range(total_blocks):
            for param in model.features[i].parameters():
                param.requires_grad = False            
        for i in range(new_frozen_blocks, prev_frozen_blocks): # note that the blocks has zero indexing
            for param in model.features[i].parameters():
                param.requires_grad = True
            unfrozen_indices.append(i)
        print(f"🟢 Intermediate stage → Newly unfrozen blocks: {unfrozen_indices}")

    # ---- Final stage: unfreeze entire backbone ----
    if final_stage:
        assert prev_unfrozen is not None, \
            "prev_unfrozen must be provided for final stage."
    
        total_blocks = len(model.features)

        prev_frozen_blocks = total_blocks - prev_unfrozen
        unfrozen_indices = list(range(prev_frozen_blocks))

        for i in unfrozen_indices:
            for param in model.features[i].parameters():
                param.requires_grad = True

        print(f"🟢 Final stage → Entire backbone unfrozen (blocks {unfrozen_indices})")

    # ---- Report trainable stats ----
    total_layers = sum(1 for p in model.parameters())
    trainable_layers = sum(1 for p in model.parameters() if p.requires_grad)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"✅ Trainable layers: {trainable_layers}/{total_layers} "
          f"({trainable_layers/total_layers*100:.2f}%)")
    print(f"✅ Trainable params: {trainable_params:,}/{total_params:,} "
          f"({trainable_params/total_params*100:.2f}%)")

    return model

In [36]:
def summarize_mobilenetv2_blocks(model: nn.Module):
    """
    Summarize blocks in a MobileNetV2 model.
    
    Returns:
        pd.DataFrame: block-level summary with counts, percentages, and cumulative stats.
    """

    # ---- Collect blocks: features + classifier ----
    blocks = []
    if hasattr(model, "features"):
        for i, block in enumerate(model.features):
            blocks.append((f"{block.__class__.__name__}_{i+1}", block, "features"))
    if hasattr(model, "classifier"):
        blocks.append((f"{model.classifier.__class__.__name__}", model.classifier, "classifier"))

    # ---- Helper functions ----
    def count_layers(module: nn.Module):
        # Count atomic submodules (ignore containers)
        return sum(1 for m in module.modules() if not isinstance(m, nn.Sequential))

    def count_params(module: nn.Module):
        return sum(p.numel() for p in module.parameters())

    def count_trainable_params(module: nn.Module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    # ---- Build dataframe rows ----
    rows = []
    for order, (name, block, module_name) in enumerate(blocks, 1):
        n_layers = count_layers(block)
        n_params = count_params(block)
        n_trainable = count_trainable_params(block)
        rows.append({
            "Block": name,
            "Order": order,
            "Module": module_name,
            "Layers (#)": n_layers,
            "Params (#)": n_params,
            "Trainable Params (#)": n_trainable,
        })

    df = pd.DataFrame(rows)

    # ---- Totals for whole model ----
    total_layers_model = df["Layers (#)"].sum()
    total_params_model = df["Params (#)"].sum()
    total_trainable_params_model = df["Trainable Params (#)"].sum()

    # Per-model percentages
    df["% Layers (Model)"] = df["Layers (#)"] / total_layers_model * 100
    df["% Params (Model)"] = df["Params (#)"] / total_params_model * 100
    df["% Trainable Params (Model)"] = df["Trainable Params (#)"] / total_params_model * 100

    # Accumulative (start from deepest → reverse order)
    df["Cumulative Layers (Model)"] = df["Layers (#)"][::-1].cumsum()[::-1]
    df["Cumulative % Layers (Model)"] = df["Cumulative Layers (Model)"] / total_layers_model * 100
    df["Cumulative Params (Model)"] = df["Params (#)"][::-1].cumsum()[::-1]
    df["Cumulative % Params (Model)"] = df["Cumulative Params (Model)"] / total_params_model * 100
    # ---- Per-module stats ----
    module_stats = []
    for module_name in df["Module"].unique():
        sub = df[df["Module"] == module_name]
        total_layers_mod = sub["Layers (#)"].sum()
        total_params_mod = sub["Params (#)"].sum()
        total_trainable_mod = sub["Trainable Params (#)"].sum()
        for idx in sub.index:
            module_stats.append({
                "Layers (Module %)": df.loc[idx, "Layers (#)"] / total_layers_mod * 100,
                "Params (Module %)": df.loc[idx, "Params (#)"] / total_params_mod * 100,
                "Trainable Params (Module %)": df.loc[idx, "Trainable Params (#)"] / total_trainable_mod * 100 if total_trainable_mod > 0 else 0,
                "Cumulative Layers (Module)": sub["Layers (#)"][::-1].cumsum()[::-1].loc[idx],
                "Cumulative % Layers (Module)": sub["Layers (#)"][::-1].cumsum()[::-1].loc[idx] / total_layers_mod * 100,
                "Cumulative Params (Module)": sub["Params (#)"][::-1].cumsum()[::-1].loc[idx],
                "Cumulative % Params (Module)": sub["Params (#)"][::-1].cumsum()[::-1].loc[idx] / total_params_mod * 100,
            })
    module_df = pd.DataFrame(module_stats, index=df.index)

    # Combine
    df = pd.concat([df, module_df], axis=1)

    return df

## 3. Model Training

In [37]:
def train_model(model, train_loader, criterion, optimizer, class_names, epoch_num=1, log_interval=100):
    start_time = time.time()  # Start tracking time
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")  # Use GPU if available on Mac

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start_time = time.time()  # Track epoch time

    for batch_idx, (images, labels) in enumerate(train_loader, 1):  # Start index from 1
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Compute accuracy
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        accuracy = 100 * correct / total

        # Store predictions for per-class metrics
        for label, pred in zip(labels.cpu().numpy(), predicted.cpu().numpy()):
            class_metrics[label]["correct"] += (label == pred)
            class_metrics[label]["total"] += 1
            class_metrics[label]["y_true"].append(label)
            class_metrics[label]["y_pred"].append(pred)

        # Print loss and accuracy every 100 batches
        if batch_idx % log_interval == 0:
            elapsed_time = (time.time() - start_time)/60
            print(f"Epoch {epoch_num}, Batch {batch_idx}, Loss: {running_loss/batch_idx:.4f}, "
                  f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    epoch_time = (time.time() - epoch_start_time)/60
    print(f"Epoch {epoch_num} Completed - Average Loss: {running_loss/len(train_loader):.4f}, "
          f"Accuracy: {accuracy:.2f}%, Epoch Time: {epoch_time:.2f}m")
    overall_accuracy = 100 * correct / total

    total_time = (time.time() - start_time)/60
    print(f"Training Complete Epoch {epoch_num} - Total Time: {total_time:.2f}m")

    # Calculate per-class accuracy, precision, recall, and F1-score
    all_y_true = []
    all_y_pred = []
    
    for class_idx in sorted(class_metrics.keys()):
        all_y_true.extend(class_metrics[class_idx]["y_true"])
        all_y_pred.extend(class_metrics[class_idx]["y_pred"])
    
    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class results in alignment with unique_classes
    class_results = []
    for class_idx in unique_classes:
        class_results.append([
            class_names[class_idx],
            per_class_metrics["precision"][unique_classes.index(class_idx)]*100,
            per_class_metrics["recall"][unique_classes.index(class_idx)]*100,
            per_class_metrics["f1_score"][unique_classes.index(class_idx)]*100
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()*100
    macro_recall = per_class_metrics["recall"].mean()*100
    macro_f1 = per_class_metrics["f1_score"].mean()*100

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    avg_loss = running_loss / len(train_loader)

    return {
    'model': model,
    'optimizer': optimizer,
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'total_loss': running_loss,
    'avg_loss': avg_loss,
    'epoch_time': epoch_time
            }

## 4. Model Evaluation

In [38]:
def evaluate_model(model, val_loader, criterion, class_names, epoch_num=1, log_interval=100):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")
    
    start_time = time.time()  # Track evaluation time

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})
    all_y_true = []
    all_y_pred = []
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(val_loader, 1):  # Start index from 1
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)

            all_y_true.extend(labels.cpu().numpy())
            all_y_pred.extend(preds.cpu().numpy())

            # Compute accuracy
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            accuracy = 100 * correct / total

            # Store per-class metrics
            for label, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                class_metrics[label]["correct"] += (label == pred)
                class_metrics[label]["total"] += 1
                class_metrics[label]["y_true"].append(label)
                class_metrics[label]["y_pred"].append(pred)

            # Print progress every 100 batches
            if batch_idx % log_interval == 0:
                elapsed_time = (time.time() - start_time)/60
                avg_loss = running_loss / batch_idx
                print(f"Batch {batch_idx}, Loss: {avg_loss:.4f}, "
                      f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    total_time = (time.time() - start_time)/60
    avg_loss = running_loss / len(val_loader)
    print(f"Evaluation Complete - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%, Total Time: {total_time:.2f}m")
    overall_accuracy = 100 * correct / total

    # Compute per-class accuracy
    class_results = []
    for class_idx in sorted(class_metrics.keys()):
        metrics = class_metrics[class_idx]
        class_results.append([class_names[class_idx]])

    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class metrics
    for idx, class_label in enumerate(unique_classes):
        class_results[idx].extend([
            per_class_metrics["precision"][idx],
            per_class_metrics["recall"][idx],
            per_class_metrics["f1_score"][idx]
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()
    macro_recall = per_class_metrics["recall"].mean()
    macro_f1 = per_class_metrics["f1_score"].mean()

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    return {
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'avg_loss': avg_loss,
    'epoch_time': total_time
            }

## 5. Create performance summary

In [39]:
def create_performance_summary(df, micro_accuracy, avg_loss, model_name, train_val, class_weights, time_taken, epoch):
    # Extract macro metrics from the last row
    
    macro_row = df[df['Species'] == 'Average Performance (Macro)']
    precision_macro = macro_row['Precision'].values[0]
    recall_macro = macro_row['Recall'].values[0]
    f1_macro = macro_row['F1-score'].values[0]


    # Exclude the last row to analyze per-class accuracies
    class_accuracies = df[df['Species'] != 'Average Performance (Macro)'].copy()

    # Identify best and worst class based on accuracy
    best_class = class_accuracies.loc[class_accuracies['F1-score'].idxmax(), 'Species']
    worst_class = class_accuracies.loc[class_accuracies['F1-score'].idxmin(), 'Species']

    # Compute the median accuracy and find the nearest class
    median_f1_macro = class_accuracies['F1-score'].median()
    class_accuracies['Abs_Diff'] = (class_accuracies['F1-score'] - median_f1_macro).abs()
    median_class = class_accuracies.loc[class_accuracies['Abs_Diff'].idxmin(), 'Species']

    # Compute the number of classes above micro and macro accuracy averages
    num_classes_above_f1_macro_avg = (class_accuracies['F1-score'] > f1_macro).sum()
    num_classes_above_f1_macro_median = (class_accuracies['F1-score'] > median_f1_macro).sum()

    # Create the final summary DataFrame
    performance_summary = pd.DataFrame({
        "Model": [model_name],
        "Train & Validation": [train_val],
        "Accuracy (micro)": [micro_accuracy],
        "Precision (macro)": [precision_macro],
        "Recall (macro)": [recall_macro],
        "F1 (macro)": [f1_macro],
        "Loss": [avg_loss],
        "Class_weights": [class_weights],
        "Best_class": [best_class],
        "Median_class": [median_class],
        "Worst_class": [worst_class],
        "num_classes_above_f1_macro_avg": [num_classes_above_f1_macro_avg],
        "num_classes_above_f1_macro_median": [num_classes_above_f1_macro_median],
        "Time Taken (mins)": [time_taken],
        "Epoch": [epoch]
    })


    return performance_summary

## 6. Model Training & Evaluation Loop

In [40]:
def train_and_evaluate_model(
    model: nn.Module,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    class_names,
    start_epoch: int,
    end_epoch: int,
    model_name: str,
    save_dir: str,
    class_weights: str,
    log_interval: int,
    evaluate = True
):
    """
    Train and evaluate a model, storing per-epoch metrics, summaries, and states.

    Returns:
        dict: {
            'epoch_states': {epoch_num: {'model_state': ..., 'optimizer_state': ..., 'train_metrics': ..., 'val_metrics': ...}},
            'train_results_df': pd.DataFrame,
            'train_summary_df': pd.DataFrame,
            'val_results_df': pd.DataFrame,
            'val_summary_df': pd.DataFrame
        }
    """
    device = torch.device("mps")
    model.to(device)  # Move model to MPS

    # Initialize storage
    epoch_states = {}
    train_results_list, train_summary_list = [], []
    val_results_list, val_summary_list = [], []


    for epoch in range(start_epoch, end_epoch + 1):
        print(f"\n🚀 Epoch {epoch} - Training Started...\n")
        
        # ---- Training ----
        training_result = train_model(
            model, train_loader, criterion, optimizer, class_names, epoch_num=epoch, log_interval=log_interval
        )
        train_summary = create_performance_summary(
            training_result['metrics_df'],
            training_result['overall_accuracy'],
            training_result['avg_loss'],
            model_name=model_name,
            train_val='Train',
            class_weights=class_weights,
            time_taken=training_result['epoch_time'],
            epoch = epoch
        )
        
        train_results_list.append(training_result['metrics_df'])
        train_summary_list.append(train_summary)
        if evaluate:

            print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
            # ---- Validation ----
            val_result = evaluate_model(
                model, val_loader, criterion, class_names, epoch_num=epoch, log_interval=log_interval
            )
            val_summary = create_performance_summary(
                val_result['metrics_df'],
                val_result['overall_accuracy'],
                val_result['avg_loss'],
                model_name=model_name,
                train_val='Validation',
                class_weights=class_weights,
                time_taken=val_result['epoch_time'],
                epoch = epoch
            )
    
            val_results_list.append(val_result['metrics_df'])
            val_summary_list.append(val_summary)
            print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

        else:
            print(f"\n✅ Epoch {epoch} - Training Completed (No Evaluation).\n")

        # ---- Save model + optimizer state for this epoch ----
        epoch_states[epoch] = {
            'model_state_dict': copy.deepcopy(model.state_dict()),
            'optimizer_state_dict': copy.deepcopy(optimizer.state_dict())
        }

        

    # ---- Combine DataFrames ----
    train_results_df = pd.concat(train_results_list, ignore_index=True)
    train_summary_df = pd.concat(train_summary_list, ignore_index=True)
    if evaluate:
        val_results_df = pd.concat(val_results_list, ignore_index=True)
        val_summary_df = pd.concat(val_summary_list, ignore_index=True)

    else:
        val_results_df, val_summary_df = None, None

    # Optionally save to CSV (you can remove if not needed)
    train_results_df.to_csv(f"{save_dir}/{model_name}_train_results{start_epoch}_{end_epoch}.csv", index=False)
    train_summary_df.to_csv(f"{save_dir}/{model_name}_train_summary{start_epoch}_{end_epoch}.csv", index=False)
    if evaluate:
        val_results_df.to_csv(f"{save_dir}/{model_name}_val_results{start_epoch}_{end_epoch}.csv", index=False)
        val_summary_df.to_csv(f"{save_dir}/{model_name}_val_summary{start_epoch}_{end_epoch}.csv", index=False)

    return epoch_states, {
        'train_results_df': train_results_df,
        'train_summary_df': train_summary_df,
        'val_results_df': val_results_df,
        'val_summary_df': val_summary_df
    }

# <center>**MobileNetV2 staged training**</center>

## **Stage A**

In [42]:
mobilenetv2_stageA = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
mobilenetv2_stageA = configure_mobilenetv2_training(mobilenetv2_stageA,
    NUM_CLASSES,
    stage_a = True)
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageA)
df_summary

🟢 Stage A → Classifier head is trainable, backbone frozen.
✅ Trainable layers: 2/158 (1.27%)
✅ Trainable params: 42,273/2,266,145 (1.87%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,0,1.898734,0.040951,0.000000,158,100.000000,2266145,100.000000,1.923077,0.041729,0.0,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,0,3.797468,0.039539,0.000000,155,98.101266,2265217,99.959049,3.846154,0.040290,0.0,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,0,5.696203,0.226640,0.000000,149,94.303797,2264321,99.919511,5.769231,0.230949,0.0,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,0,5.696203,0.389737,0.000000,140,88.607595,2259185,99.692870,5.769231,0.397145,0.0,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,0,5.696203,0.441278,0.000000,131,82.911392,2250353,99.303134,5.769231,0.449666,0.0,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,0,5.696203,0.655210,0.000000,122,77.215190,2240353,98.861856,5.769231,0.667664,0.0,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,0,5.696203,0.655210,0.000000,113,71.518987,2225505,98.206646,5.769231,0.667664,0.0,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,0,5.696203,0.929155,0.000000,104,65.822785,2210657,97.551436,5.769231,0.946817,0.0,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,0,5.696203,2.394904,0.000000,95,60.126582,2189601,96.622281,5.769231,2.440428,0.0,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,0,5.696203,2.394904,0.000000,86,54.430380,2135329,94.227377,5.769231,2.440428,0.0,84,53.846154,2093056,94.117647


In [43]:
inverse_frequency_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=False)
inverse_frequency_criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights)
stageA_optimizer = torch.optim.AdamW(mobilenetv2_stageA.parameters(), lr=1e-3,
                            weight_decay=1e-4, betas=(0.9, 0.999))
display(train_dataset.transform)
val_dataset.transform

Compose(
      ToImage()
      RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomVerticalFlip(p=0.2)
      RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
      ToDtype(scale=True)
      RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

Compose(
      ToImage()
      Resize(size=[256], interpolation=InterpolationMode.BILINEAR, antialias=True)
      CenterCrop(size=(224, 224))
      ToDtype(scale=True)
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)

In [39]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageA, train_loader, val_loader, inverse_frequency_criterion, stageA_optimizer, train_dataset.classes,
                         start_epoch = 1, end_epoch = 5, model_name = 'mobilenetv2_stageA', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 1 - Training Started...

Epoch 1, Batch 100, Loss: 2.9786, Accuracy: 24.62%, Time Passed: 2.87m
Epoch 1, Batch 200, Loss: 2.5948, Accuracy: 36.22%, Time Passed: 5.87m
Epoch 1, Batch 300, Loss: 2.3455, Accuracy: 42.00%, Time Passed: 8.87m
Epoch 1, Batch 400, Loss: 2.1701, Accuracy: 46.20%, Time Passed: 11.87m
Epoch 1, Batch 500, Loss: 2.0355, Accuracy: 49.11%, Time Passed: 14.91m
Epoch 1 Completed - Average Loss: 1.9867, Accuracy: 50.11%, Epoch Time: 16.18m
Training Complete Epoch 1 - Total Time: 16.18m

✅ Epoch 1 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.8591, Accuracy: 44.31%, Time Passed: 3.71m
Evaluation Complete - Loss: 1.7239, Accuracy: 50.75%, Total Time: 6.65m

📊 Epoch 1 - Evaluation Completed.


🚀 Epoch 2 - Training Started...

Epoch 2, Batch 100, Loss: 1.3919, Accuracy: 61.94%, Time Passed: 3.11m
Epoch 2, Batch 200, Loss: 1.3456, Accuracy: 63.41%, Time Passed: 6.17m
Epoch 2, Batch 300, Loss: 1.3183, Accuracy: 64.14%, Time Passed: 9.23m
Epoch 2, 

In [40]:
start_epoch = 1
end_epoch = 5

In [41]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [42]:
del training_states

In [45]:
start_epoch = 6
end_epoch = 10
start_epoch, end_epoch

(6, 10)

In [46]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageA, train_loader, val_loader, inverse_frequency_criterion, stageA_optimizer, train_dataset.classes,
                         start_epoch = start_epoch, end_epoch = end_epoch, model_name = 'mobilenetv2_stageA', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 6 - Training Started...

Epoch 6, Batch 100, Loss: 0.9055, Accuracy: 71.47%, Time Passed: 2.88m
Epoch 6, Batch 200, Loss: 0.9044, Accuracy: 72.17%, Time Passed: 5.96m
Epoch 6, Batch 300, Loss: 0.9023, Accuracy: 72.15%, Time Passed: 9.18m
Epoch 6, Batch 400, Loss: 0.9054, Accuracy: 72.05%, Time Passed: 12.39m
Epoch 6, Batch 500, Loss: 0.9067, Accuracy: 72.06%, Time Passed: 15.65m
Epoch 6 Completed - Average Loss: 0.9035, Accuracy: 72.16%, Epoch Time: 17.08m
Training Complete Epoch 6 - Total Time: 17.08m

✅ Epoch 6 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.4349, Accuracy: 55.66%, Time Passed: 4.49m
Evaluation Complete - Loss: 1.2934, Accuracy: 60.77%, Total Time: 8.03m

📊 Epoch 6 - Evaluation Completed.


🚀 Epoch 7 - Training Started...

Epoch 7, Batch 100, Loss: 0.9111, Accuracy: 72.28%, Time Passed: 4.07m
Epoch 7, Batch 200, Loss: 0.8960, Accuracy: 72.17%, Time Passed: 8.02m
Epoch 7, Batch 300, Loss: 0.8808, Accuracy: 72.42%, Time Passed: 11.87m
Epoch 7,

In [49]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [52]:
del training_states

In [53]:
start_epoch = 11
end_epoch = 20
start_epoch, end_epoch

(11, 20)

In [54]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageA, train_loader, val_loader, inverse_frequency_criterion, stageA_optimizer, train_dataset.classes,
                         start_epoch = start_epoch, end_epoch = end_epoch, model_name = 'mobilenetv2_stageA', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 11 - Training Started...

Epoch 11, Batch 100, Loss: 0.7804, Accuracy: 74.62%, Time Passed: 3.20m
Epoch 11, Batch 200, Loss: 0.8043, Accuracy: 74.05%, Time Passed: 6.42m
Epoch 11, Batch 300, Loss: 0.8107, Accuracy: 73.97%, Time Passed: 10.14m
Epoch 11, Batch 400, Loss: 0.8074, Accuracy: 74.05%, Time Passed: 13.33m
Epoch 11, Batch 500, Loss: 0.7952, Accuracy: 74.28%, Time Passed: 16.55m
Epoch 11 Completed - Average Loss: 0.7952, Accuracy: 74.30%, Epoch Time: 17.83m
Training Complete Epoch 11 - Total Time: 17.83m

✅ Epoch 11 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.3170, Accuracy: 59.59%, Time Passed: 4.02m
Evaluation Complete - Loss: 1.1785, Accuracy: 64.25%, Total Time: 7.10m

📊 Epoch 11 - Evaluation Completed.


🚀 Epoch 12 - Training Started...

Epoch 12, Batch 100, Loss: 0.7754, Accuracy: 75.25%, Time Passed: 3.73m
Epoch 12, Batch 200, Loss: 0.7696, Accuracy: 75.45%, Time Passed: 7.91m
Epoch 12, Batch 300, Loss: 0.7790, Accuracy: 75.07%, Time Passed: 

In [55]:
start_epoch, end_epoch

(11, 20)

In [56]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [57]:
del training_states

## **Stage B**

In [58]:
mobilenetv2_stageB = configure_mobilenetv2_training(
                                                    mobilenetv2_stageA,
                                                    NUM_CLASSES, intermediate=True,
                                                    intermediate_blocks = [0, 2]
                                                   )
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageB)
df_summary

🟢 Intermediate stage → Newly unfrozen blocks: [17, 18]
✅ Trainable layers: 14/158 (8.86%)
✅ Trainable params: 928,353/2,266,145 (40.97%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,0,1.898734,0.040951,0.000000,158,100.000000,2266145,100.000000,1.923077,0.041729,0.000000,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,0,3.797468,0.039539,0.000000,155,98.101266,2265217,99.959049,3.846154,0.040290,0.000000,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,0,5.696203,0.226640,0.000000,149,94.303797,2264321,99.919511,5.769231,0.230949,0.000000,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,0,5.696203,0.389737,0.000000,140,88.607595,2259185,99.692870,5.769231,0.397145,0.000000,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,0,5.696203,0.441278,0.000000,131,82.911392,2250353,99.303134,5.769231,0.449666,0.000000,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,0,5.696203,0.655210,0.000000,122,77.215190,2240353,98.861856,5.769231,0.667664,0.000000,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,0,5.696203,0.655210,0.000000,113,71.518987,2225505,98.206646,5.769231,0.667664,0.000000,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,0,5.696203,0.929155,0.000000,104,65.822785,2210657,97.551436,5.769231,0.946817,0.000000,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,0,5.696203,2.394904,0.000000,95,60.126582,2189601,96.622281,5.769231,2.440428,0.000000,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,0,5.696203,2.394904,0.000000,86,54.430380,2135329,94.227377,5.769231,2.440428,0.000000,84,53.846154,2093056,94.117647


In [46]:
stageB_optimizer = torch.optim.AdamW(mobilenetv2_stageB.parameters(), lr=1e-4,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [60]:
start_epoch = 21
end_epoch = 25
start_epoch, end_epoch

(21, 25)

In [61]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageB, train_loader, val_loader, inverse_frequency_criterion, stageB_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageB', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 21 - Training Started...

Epoch 21, Batch 100, Loss: 0.6929, Accuracy: 77.09%, Time Passed: 2.96m
Epoch 21, Batch 200, Loss: 0.6839, Accuracy: 76.98%, Time Passed: 6.03m
Epoch 21, Batch 300, Loss: 0.6725, Accuracy: 77.69%, Time Passed: 9.20m
Epoch 21, Batch 400, Loss: 0.6658, Accuracy: 77.76%, Time Passed: 12.35m
Epoch 21, Batch 500, Loss: 0.6623, Accuracy: 77.83%, Time Passed: 15.62m
Epoch 21 Completed - Average Loss: 0.6580, Accuracy: 78.05%, Epoch Time: 17.01m
Training Complete Epoch 21 - Total Time: 17.01m

✅ Epoch 21 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.2619, Accuracy: 62.09%, Time Passed: 3.99m
Evaluation Complete - Loss: 1.1210, Accuracy: 66.79%, Total Time: 6.84m

📊 Epoch 21 - Evaluation Completed.


🚀 Epoch 22 - Training Started...

Epoch 22, Batch 100, Loss: 0.6043, Accuracy: 79.84%, Time Passed: 3.45m
Epoch 22, Batch 200, Loss: 0.5884, Accuracy: 80.45%, Time Passed: 7.20m
Epoch 22, Batch 300, Loss: 0.5781, Accuracy: 80.65%, Time Passed: 1

In [62]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [63]:
del training_states

In [64]:
start_epoch = 26
end_epoch = 31
start_epoch, end_epoch

(26, 31)

In [65]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageB, train_loader, val_loader, inverse_frequency_criterion, stageB_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageB', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 26 - Training Started...

Epoch 26, Batch 100, Loss: 0.3965, Accuracy: 86.25%, Time Passed: 3.47m
Epoch 26, Batch 200, Loss: 0.3977, Accuracy: 86.34%, Time Passed: 6.82m
Epoch 26, Batch 300, Loss: 0.4017, Accuracy: 86.11%, Time Passed: 10.35m
Epoch 26, Batch 400, Loss: 0.4015, Accuracy: 85.95%, Time Passed: 14.09m
Epoch 26, Batch 500, Loss: 0.4003, Accuracy: 85.94%, Time Passed: 17.66m
Epoch 26 Completed - Average Loss: 0.4037, Accuracy: 85.75%, Epoch Time: 19.14m
Training Complete Epoch 26 - Total Time: 19.14m

✅ Epoch 26 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.1000, Accuracy: 67.69%, Time Passed: 4.72m
Evaluation Complete - Loss: 0.9549, Accuracy: 72.46%, Total Time: 7.92m

📊 Epoch 26 - Evaluation Completed.


🚀 Epoch 27 - Training Started...

Epoch 27, Batch 100, Loss: 0.3599, Accuracy: 86.78%, Time Passed: 3.67m
Epoch 27, Batch 200, Loss: 0.3541, Accuracy: 87.38%, Time Passed: 7.10m
Epoch 27, Batch 300, Loss: 0.3687, Accuracy: 87.01%, Time Passed: 

In [69]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [70]:
del training_states

In [77]:
start_epoch = 32
end_epoch = 35
start_epoch, end_epoch

(32, 35)

In [78]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageB, train_loader, val_loader, inverse_frequency_criterion, stageB_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageB', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 32 - Training Started...

Epoch 32, Batch 100, Loss: 0.3357, Accuracy: 88.59%, Time Passed: 3.02m
Epoch 32, Batch 200, Loss: 0.3120, Accuracy: 89.44%, Time Passed: 6.13m
Epoch 32, Batch 300, Loss: 0.3022, Accuracy: 89.60%, Time Passed: 9.37m
Epoch 32, Batch 400, Loss: 0.3065, Accuracy: 89.45%, Time Passed: 12.67m
Epoch 32, Batch 500, Loss: 0.3046, Accuracy: 89.47%, Time Passed: 16.01m
Epoch 32 Completed - Average Loss: 0.3027, Accuracy: 89.50%, Epoch Time: 17.52m
Training Complete Epoch 32 - Total Time: 17.52m

✅ Epoch 32 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.9411, Accuracy: 72.44%, Time Passed: 4.26m
Evaluation Complete - Loss: 0.8768, Accuracy: 75.77%, Total Time: 7.20m

📊 Epoch 32 - Evaluation Completed.


🚀 Epoch 33 - Training Started...

Epoch 33, Batch 100, Loss: 0.3132, Accuracy: 89.22%, Time Passed: 3.69m
Epoch 33, Batch 200, Loss: 0.3026, Accuracy: 89.52%, Time Passed: 7.41m
Epoch 33, Batch 300, Loss: 0.2970, Accuracy: 89.77%, Time Passed: 1

In [79]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [84]:
start_epoch = 36
end_epoch = 40
start_epoch, end_epoch

(36, 40)

In [85]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageB, train_loader, val_loader, inverse_frequency_criterion, stageB_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageB', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 36 - Training Started...

Epoch 36, Batch 100, Loss: 0.2493, Accuracy: 90.78%, Time Passed: 2.96m
Epoch 36, Batch 200, Loss: 0.2590, Accuracy: 90.78%, Time Passed: 6.00m
Epoch 36, Batch 300, Loss: 0.2525, Accuracy: 90.99%, Time Passed: 9.17m
Epoch 36, Batch 400, Loss: 0.2605, Accuracy: 90.71%, Time Passed: 12.46m
Epoch 36, Batch 500, Loss: 0.2638, Accuracy: 90.69%, Time Passed: 15.76m
Epoch 36 Completed - Average Loss: 0.2628, Accuracy: 90.66%, Epoch Time: 17.24m
Training Complete Epoch 36 - Total Time: 17.24m

✅ Epoch 36 - Training Completed. Starting Evaluation...

Batch 100, Loss: 1.0056, Accuracy: 71.03%, Time Passed: 4.06m
Evaluation Complete - Loss: 0.9070, Accuracy: 75.12%, Total Time: 6.90m

📊 Epoch 36 - Evaluation Completed.


🚀 Epoch 37 - Training Started...

Epoch 37, Batch 100, Loss: 0.2430, Accuracy: 91.22%, Time Passed: 3.28m
Epoch 37, Batch 200, Loss: 0.2551, Accuracy: 90.59%, Time Passed: 6.53m
Epoch 37, Batch 300, Loss: 0.2553, Accuracy: 90.60%, Time Passed: 9

In [87]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

## **Stage C**

### **Checkpoint Validation**

In [89]:
mobilenetv2_stageC = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageC.to(device)
mobilenetv2_stageC.load_state_dict(training_states[39]["model_state_dict"])

<All keys matched successfully>

In [91]:
_ = evaluate_model(
            mobilenetv2_stageC, val_loader, inverse_frequency_criterion, train_dataset.classes, epoch_num=39, log_interval=100)

Batch 100, Loss: 0.9590, Accuracy: 72.50%, Time Passed: 3.60m
Evaluation Complete - Loss: 0.8577, Accuracy: 76.37%, Total Time: 6.31m


In [92]:
del training_states

### **Training**

In [95]:
mobilenetv2_stageC = configure_mobilenetv2_training(
                                                    mobilenetv2_stageC,
                                                    NUM_CLASSES, intermediate=True,
                                                    intermediate_blocks = [0, 4]
                                                   )
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageC)
df_summary

🟢 Intermediate stage → Newly unfrozen blocks: [15, 16, 17, 18]
✅ Trainable layers: 32/158 (20.25%)
✅ Trainable params: 1,568,353/2,266,145 (69.21%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,0,1.898734,0.040951,0.000000,158,100.000000,2266145,100.000000,1.923077,0.041729,0.000000,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,0,3.797468,0.039539,0.000000,155,98.101266,2265217,99.959049,3.846154,0.040290,0.000000,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,0,5.696203,0.226640,0.000000,149,94.303797,2264321,99.919511,5.769231,0.230949,0.000000,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,0,5.696203,0.389737,0.000000,140,88.607595,2259185,99.692870,5.769231,0.397145,0.000000,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,0,5.696203,0.441278,0.000000,131,82.911392,2250353,99.303134,5.769231,0.449666,0.000000,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,0,5.696203,0.655210,0.000000,122,77.215190,2240353,98.861856,5.769231,0.667664,0.000000,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,0,5.696203,0.655210,0.000000,113,71.518987,2225505,98.206646,5.769231,0.667664,0.000000,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,0,5.696203,0.929155,0.000000,104,65.822785,2210657,97.551436,5.769231,0.946817,0.000000,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,0,5.696203,2.394904,0.000000,95,60.126582,2189601,96.622281,5.769231,2.440428,0.000000,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,0,5.696203,2.394904,0.000000,86,54.430380,2135329,94.227377,5.769231,2.440428,0.000000,84,53.846154,2093056,94.117647


In [98]:
stageC_optimizer = torch.optim.AdamW(mobilenetv2_stageC.parameters(), lr=5e-5,
                            weight_decay=1e-4, betas=(0.9, 0.999))

In [99]:
start_epoch = 40
end_epoch = 50
start_epoch, end_epoch

(40, 50)

In [100]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageC, train_loader, val_loader, inverse_frequency_criterion, stageC_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageC', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 40 - Training Started...

Epoch 40, Batch 100, Loss: 0.2234, Accuracy: 91.50%, Time Passed: 3.97m
Epoch 40, Batch 200, Loss: 0.2143, Accuracy: 91.75%, Time Passed: 7.83m
Epoch 40, Batch 300, Loss: 0.2161, Accuracy: 91.92%, Time Passed: 11.52m
Epoch 40, Batch 400, Loss: 0.2180, Accuracy: 91.93%, Time Passed: 14.87m
Epoch 40, Batch 500, Loss: 0.2165, Accuracy: 92.14%, Time Passed: 18.24m
Epoch 40 Completed - Average Loss: 0.2165, Accuracy: 92.15%, Epoch Time: 19.64m
Training Complete Epoch 40 - Total Time: 19.64m

✅ Epoch 40 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.9449, Accuracy: 73.09%, Time Passed: 4.22m
Evaluation Complete - Loss: 0.8368, Accuracy: 77.25%, Total Time: 7.06m

📊 Epoch 40 - Evaluation Completed.


🚀 Epoch 41 - Training Started...

Epoch 41, Batch 100, Loss: 0.1948, Accuracy: 93.16%, Time Passed: 3.40m
Epoch 41, Batch 200, Loss: 0.2027, Accuracy: 92.88%, Time Passed: 6.82m
Epoch 41, Batch 300, Loss: 0.2186, Accuracy: 92.41%, Time Passed: 

In [103]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [104]:
del training_states

In [106]:
start_epoch = 51
end_epoch = 52
start_epoch, end_epoch

(51, 52)

In [107]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageC, train_loader, val_loader, inverse_frequency_criterion, stageC_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageC', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 51 - Training Started...

Epoch 51, Batch 100, Loss: 0.1609, Accuracy: 94.59%, Time Passed: 3.51m
Epoch 51, Batch 200, Loss: 0.1530, Accuracy: 94.59%, Time Passed: 7.09m
Epoch 51, Batch 300, Loss: 0.1527, Accuracy: 94.51%, Time Passed: 10.70m
Epoch 51, Batch 400, Loss: 0.1574, Accuracy: 94.37%, Time Passed: 14.09m
Epoch 51, Batch 500, Loss: 0.1557, Accuracy: 94.41%, Time Passed: 17.53m
Epoch 51 Completed - Average Loss: 0.1543, Accuracy: 94.48%, Epoch Time: 18.99m
Training Complete Epoch 51 - Total Time: 18.99m

✅ Epoch 51 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.8368, Accuracy: 76.47%, Time Passed: 4.24m
Evaluation Complete - Loss: 0.7574, Accuracy: 80.00%, Total Time: 7.14m

📊 Epoch 51 - Evaluation Completed.


🚀 Epoch 52 - Training Started...

Epoch 52, Batch 100, Loss: 0.1514, Accuracy: 94.53%, Time Passed: 3.51m
Epoch 52, Batch 200, Loss: 0.1476, Accuracy: 94.80%, Time Passed: 7.05m
Epoch 52, Batch 300, Loss: 0.1413, Accuracy: 94.90%, Time Passed: 

In [108]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [109]:
del training_states

In [112]:
start_epoch = 53
end_epoch = 60
start_epoch, end_epoch

(53, 60)

In [113]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageC, train_loader, val_loader, inverse_frequency_criterion, stageC_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageC', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 53 - Training Started...

Epoch 53, Batch 100, Loss: 0.1444, Accuracy: 94.06%, Time Passed: 3.32m
Epoch 53, Batch 200, Loss: 0.1500, Accuracy: 94.47%, Time Passed: 6.71m
Epoch 53, Batch 300, Loss: 0.1403, Accuracy: 94.73%, Time Passed: 10.21m
Epoch 53, Batch 400, Loss: 0.1411, Accuracy: 94.73%, Time Passed: 13.76m
Epoch 53, Batch 500, Loss: 0.1438, Accuracy: 94.83%, Time Passed: 17.51m
Epoch 53 Completed - Average Loss: 0.1432, Accuracy: 94.83%, Epoch Time: 18.93m
Training Complete Epoch 53 - Total Time: 18.93m

✅ Epoch 53 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.9112, Accuracy: 75.31%, Time Passed: 4.32m
Evaluation Complete - Loss: 0.8244, Accuracy: 78.87%, Total Time: 7.31m

📊 Epoch 53 - Evaluation Completed.


🚀 Epoch 54 - Training Started...

Epoch 54, Batch 100, Loss: 0.1340, Accuracy: 94.91%, Time Passed: 3.74m
Epoch 54, Batch 200, Loss: 0.1368, Accuracy: 94.92%, Time Passed: 7.58m
Epoch 54, Batch 300, Loss: 0.1387, Accuracy: 94.85%, Time Passed: 

In [114]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [115]:
del training_states

In [116]:
start_epoch = 61
end_epoch = 65
start_epoch, end_epoch

(61, 65)

In [117]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageC, train_loader, val_loader, inverse_frequency_criterion, stageC_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageC', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 61 - Training Started...

Epoch 61, Batch 100, Loss: 0.1081, Accuracy: 96.47%, Time Passed: 4.06m
Epoch 61, Batch 200, Loss: 0.1235, Accuracy: 95.70%, Time Passed: 8.46m
Epoch 61, Batch 300, Loss: 0.1173, Accuracy: 95.77%, Time Passed: 12.49m
Epoch 61, Batch 400, Loss: 0.1238, Accuracy: 95.64%, Time Passed: 16.33m
Epoch 61, Batch 500, Loss: 0.1212, Accuracy: 95.64%, Time Passed: 19.99m
Epoch 61 Completed - Average Loss: 0.1216, Accuracy: 95.59%, Epoch Time: 21.46m
Training Complete Epoch 61 - Total Time: 21.46m

✅ Epoch 61 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.8957, Accuracy: 75.31%, Time Passed: 4.79m
Evaluation Complete - Loss: 0.7894, Accuracy: 79.23%, Total Time: 7.88m

📊 Epoch 61 - Evaluation Completed.


🚀 Epoch 62 - Training Started...

Epoch 62, Batch 100, Loss: 0.1010, Accuracy: 96.16%, Time Passed: 3.87m
Epoch 62, Batch 200, Loss: 0.1142, Accuracy: 95.69%, Time Passed: 7.74m
Epoch 62, Batch 300, Loss: 0.1150, Accuracy: 95.85%, Time Passed: 

In [118]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [44]:
del training_states

### **Checkpoint Validation**

In [48]:
inverse_frequency_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=False)
inverse_frequency_criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights)

In [46]:
training_states = joblib.load(f'mobilenetv2/training_states_{61}_{65}.pkl')

In [42]:
mobilenetv2_stageC = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageC.to(device)
mobilenetv2_stageC.load_state_dict(training_states[65]["model_state_dict"])

<All keys matched successfully>

In [43]:
mobilenetv2_stageC = configure_mobilenetv2_training(
                                                    mobilenetv2_stageC,
                                                    NUM_CLASSES, intermediate=True,
                                                    intermediate_blocks = [0, 4]
                                                   )
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageC)
df_summary

🟢 Intermediate stage → Newly unfrozen blocks: [15, 16, 17, 18]
✅ Trainable layers: 32/158 (20.25%)
✅ Trainable params: 1,568,353/2,266,145 (69.21%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,0,1.898734,0.040951,0.000000,158,100.000000,2266145,100.000000,1.923077,0.041729,0.000000,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,0,3.797468,0.039539,0.000000,155,98.101266,2265217,99.959049,3.846154,0.040290,0.000000,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,0,5.696203,0.226640,0.000000,149,94.303797,2264321,99.919511,5.769231,0.230949,0.000000,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,0,5.696203,0.389737,0.000000,140,88.607595,2259185,99.692870,5.769231,0.397145,0.000000,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,0,5.696203,0.441278,0.000000,131,82.911392,2250353,99.303134,5.769231,0.449666,0.000000,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,0,5.696203,0.655210,0.000000,122,77.215190,2240353,98.861856,5.769231,0.667664,0.000000,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,0,5.696203,0.655210,0.000000,113,71.518987,2225505,98.206646,5.769231,0.667664,0.000000,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,0,5.696203,0.929155,0.000000,104,65.822785,2210657,97.551436,5.769231,0.946817,0.000000,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,0,5.696203,2.394904,0.000000,95,60.126582,2189601,96.622281,5.769231,2.440428,0.000000,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,0,5.696203,2.394904,0.000000,86,54.430380,2135329,94.227377,5.769231,2.440428,0.000000,84,53.846154,2093056,94.117647


In [45]:
_ = evaluate_model(
            mobilenetv2_stageC, val_loader, inverse_frequency_criterion, train_dataset.classes, epoch_num=65, log_interval=100)

Batch 100, Loss: 0.7736, Accuracy: 79.16%, Time Passed: 3.71m
Evaluation Complete - Loss: 0.7406, Accuracy: 81.37%, Total Time: 6.46m


### **Continue Training**

In [48]:
# Recreate the optimizer with the same settings as before
stageC_optimizer = torch.optim.AdamW(mobilenetv2_stageC.parameters(), lr=5e-5, weight_decay=1e-4,
                                     betas=(0.9, 0.999))

# Load the saved optimizer state
stageC_optimizer.load_state_dict(training_states[65]["optimizer_state_dict"])

In [51]:
start_epoch = 66
end_epoch = 70
start_epoch, end_epoch

(66, 70)

In [52]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageC, train_loader, val_loader, inverse_frequency_criterion, stageC_optimizer, train_dataset.classes,
                         start_epoch, end_epoch, model_name = 'mobilenetv2_stageC', save_dir = 'mobilenetv2',
                         class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 66 - Training Started...

Epoch 66, Batch 100, Loss: 0.1151, Accuracy: 95.69%, Time Passed: 2.96m
Epoch 66, Batch 200, Loss: 0.1178, Accuracy: 95.39%, Time Passed: 5.98m
Epoch 66, Batch 300, Loss: 0.1198, Accuracy: 95.49%, Time Passed: 9.09m
Epoch 66, Batch 400, Loss: 0.1155, Accuracy: 95.55%, Time Passed: 12.24m
Epoch 66, Batch 500, Loss: 0.1158, Accuracy: 95.59%, Time Passed: 15.46m
Epoch 66 Completed - Average Loss: 0.1139, Accuracy: 95.69%, Epoch Time: 16.85m
Training Complete Epoch 66 - Total Time: 16.85m

✅ Epoch 66 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.8414, Accuracy: 76.91%, Time Passed: 4.01m
Evaluation Complete - Loss: 0.7586, Accuracy: 80.42%, Total Time: 6.83m

📊 Epoch 66 - Evaluation Completed.


🚀 Epoch 67 - Training Started...

Epoch 67, Batch 100, Loss: 0.1267, Accuracy: 95.56%, Time Passed: 3.26m
Epoch 67, Batch 200, Loss: 0.1270, Accuracy: 95.41%, Time Passed: 6.91m
Epoch 67, Batch 300, Loss: 0.1287, Accuracy: 95.31%, Time Passed: 1

## **Stage D**

### **Checkpoint Validation**

In [55]:
training_states = joblib.load(f'mobilenetv2/training_states_{61}_{65}.pkl')

In [42]:
mobilenetv2_stageD = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageD.to(device)
mobilenetv2_stageD.load_state_dict(training_states[65]["model_state_dict"])

<All keys matched successfully>

In [44]:
mobilenetv2_stageD = configure_mobilenetv2_training(
                                                    mobilenetv2_stageD,
                                                    NUM_CLASSES, intermediate=True,
                                                    intermediate_blocks = [0, 6]
                                                   )
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageD)
df_summary

🟢 Intermediate stage → Newly unfrozen blocks: [13, 14, 15, 16, 17, 18]
✅ Trainable layers: 50/158 (31.65%)
✅ Trainable params: 1,841,889/2,266,145 (81.28%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,0,1.898734,0.040951,0.000000,158,100.000000,2266145,100.000000,1.923077,0.041729,0.000000,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,0,3.797468,0.039539,0.000000,155,98.101266,2265217,99.959049,3.846154,0.040290,0.000000,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,0,5.696203,0.226640,0.000000,149,94.303797,2264321,99.919511,5.769231,0.230949,0.000000,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,0,5.696203,0.389737,0.000000,140,88.607595,2259185,99.692870,5.769231,0.397145,0.000000,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,0,5.696203,0.441278,0.000000,131,82.911392,2250353,99.303134,5.769231,0.449666,0.000000,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,0,5.696203,0.655210,0.000000,122,77.215190,2240353,98.861856,5.769231,0.667664,0.000000,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,0,5.696203,0.655210,0.000000,113,71.518987,2225505,98.206646,5.769231,0.667664,0.000000,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,0,5.696203,0.929155,0.000000,104,65.822785,2210657,97.551436,5.769231,0.946817,0.000000,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,0,5.696203,2.394904,0.000000,95,60.126582,2189601,96.622281,5.769231,2.440428,0.000000,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,0,5.696203,2.394904,0.000000,86,54.430380,2135329,94.227377,5.769231,2.440428,0.000000,84,53.846154,2093056,94.117647


In [59]:
_ = evaluate_model(
            mobilenetv2_stageD, val_loader, inverse_frequency_criterion, train_dataset.classes, epoch_num=65, log_interval=100)

Batch 100, Loss: 0.7736, Accuracy: 79.16%, Time Passed: 3.77m
Evaluation Complete - Loss: 0.7406, Accuracy: 81.37%, Total Time: 6.55m


### **Continue Training**

In [45]:
stageD_optimizer = torch.optim.AdamW(
    mobilenetv2_stageD.parameters(),
    lr=3e-5,    # even smaller LR, since more params unfrozen
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

In [66]:
start_epoch = 66
end_epoch = 70
start_epoch, end_epoch

(66, 70)

In [67]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 66 - Training Started...

Epoch 66, Batch 100, Loss: 0.1146, Accuracy: 95.84%, Time Passed: 3.30m
Epoch 66, Batch 200, Loss: 0.1142, Accuracy: 95.81%, Time Passed: 6.71m
Epoch 66, Batch 300, Loss: 0.1078, Accuracy: 96.09%, Time Passed: 10.13m
Epoch 66, Batch 400, Loss: 0.1066, Accuracy: 96.12%, Time Passed: 13.61m
Epoch 66, Batch 500, Loss: 0.1107, Accuracy: 95.99%, Time Passed: 17.09m
Epoch 66 Completed - Average Loss: 0.1128, Accuracy: 95.91%, Epoch Time: 18.61m
Training Complete Epoch 66 - Total Time: 18.61m

✅ Epoch 66 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.8173, Accuracy: 77.78%, Time Passed: 4.03m
Evaluation Complete - Loss: 0.7495, Accuracy: 80.77%, Total Time: 6.88m

📊 Epoch 66 - Evaluation Completed.


🚀 Epoch 67 - Training Started...

Epoch 67, Batch 100, Loss: 0.1160, Accuracy: 96.00%, Time Passed: 3.39m
Epoch 67, Batch 200, Loss: 0.1068, Accuracy: 96.41%, Time Passed: 6.80m
Epoch 67, Batch 300, Loss: 0.1103, Accuracy: 96.20%, Time Passed: 

In [70]:
end_epoch

70

In [71]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}DDDD.pkl')

In [72]:
del training_states

In [74]:
start_epoch = 71
end_epoch = 72
start_epoch, end_epoch

(71, 72)

In [75]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 71 - Training Started...

Epoch 71, Batch 100, Loss: 0.1129, Accuracy: 95.84%, Time Passed: 3.24m
Epoch 71, Batch 200, Loss: 0.1020, Accuracy: 96.33%, Time Passed: 6.72m
Epoch 71, Batch 300, Loss: 0.0957, Accuracy: 96.34%, Time Passed: 10.15m
Epoch 71, Batch 400, Loss: 0.0998, Accuracy: 96.21%, Time Passed: 13.73m
Epoch 71, Batch 500, Loss: 0.1006, Accuracy: 96.16%, Time Passed: 17.24m
Epoch 71 Completed - Average Loss: 0.1009, Accuracy: 96.15%, Epoch Time: 18.64m
Training Complete Epoch 71 - Total Time: 18.64m

✅ Epoch 71 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7871, Accuracy: 78.97%, Time Passed: 3.98m
Evaluation Complete - Loss: 0.7560, Accuracy: 81.19%, Total Time: 6.74m

📊 Epoch 71 - Evaluation Completed.


🚀 Epoch 72 - Training Started...

Epoch 72, Batch 100, Loss: 0.1007, Accuracy: 96.56%, Time Passed: 3.27m
Epoch 72, Batch 200, Loss: 0.0985, Accuracy: 96.62%, Time Passed: 6.52m
Epoch 72, Batch 300, Loss: 0.0940, Accuracy: 96.56%, Time Passed: 

In [76]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}DDDD.pkl')

- The DDDD suffix was used as an internal filename marker to distinguish Stage D checkpoints from Stage C checkpoints covering the same epoch range.

In [77]:
del training_states

In [52]:
start_epoch = 73
end_epoch = 80
start_epoch, end_epoch

(73, 80)

In [53]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 73 - Training Started...

Epoch 73, Batch 100, Loss: 0.0853, Accuracy: 96.84%, Time Passed: 3.10m
Epoch 73, Batch 200, Loss: 0.0963, Accuracy: 96.44%, Time Passed: 6.39m
Epoch 73, Batch 300, Loss: 0.0934, Accuracy: 96.62%, Time Passed: 9.59m
Epoch 73, Batch 400, Loss: 0.0968, Accuracy: 96.47%, Time Passed: 12.80m
Epoch 73, Batch 500, Loss: 0.0962, Accuracy: 96.45%, Time Passed: 16.11m
Epoch 73 Completed - Average Loss: 0.0955, Accuracy: 96.43%, Epoch Time: 17.65m
Training Complete Epoch 73 - Total Time: 17.65m

✅ Epoch 73 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.8855, Accuracy: 76.88%, Time Passed: 4.38m
Evaluation Complete - Loss: 0.8127, Accuracy: 79.92%, Total Time: 7.32m

📊 Epoch 73 - Evaluation Completed.


🚀 Epoch 74 - Training Started...

Epoch 74, Batch 100, Loss: 0.0913, Accuracy: 96.53%, Time Passed: 3.33m
Epoch 74, Batch 200, Loss: 0.0919, Accuracy: 96.56%, Time Passed: 6.57m
Epoch 74, Batch 300, Loss: 0.0958, Accuracy: 96.40%, Time Passed: 9

In [54]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [55]:
del training_states

In [57]:
start_epoch = 81
end_epoch = 83
start_epoch, end_epoch

(81, 83)

In [58]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 81 - Training Started...

Epoch 81, Batch 100, Loss: 0.0758, Accuracy: 97.16%, Time Passed: 3.25m
Epoch 81, Batch 200, Loss: 0.0744, Accuracy: 97.09%, Time Passed: 6.61m
Epoch 81, Batch 300, Loss: 0.0812, Accuracy: 96.91%, Time Passed: 10.18m
Epoch 81, Batch 400, Loss: 0.0809, Accuracy: 97.07%, Time Passed: 13.73m
Epoch 81, Batch 500, Loss: 0.0811, Accuracy: 97.03%, Time Passed: 17.43m
Epoch 81 Completed - Average Loss: 0.0812, Accuracy: 97.05%, Epoch Time: 18.93m
Training Complete Epoch 81 - Total Time: 18.93m

✅ Epoch 81 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7886, Accuracy: 78.78%, Time Passed: 4.17m
Evaluation Complete - Loss: 0.6984, Accuracy: 82.10%, Total Time: 7.03m

📊 Epoch 81 - Evaluation Completed.


🚀 Epoch 82 - Training Started...

Epoch 82, Batch 100, Loss: 0.0813, Accuracy: 97.25%, Time Passed: 3.34m
Epoch 82, Batch 200, Loss: 0.0834, Accuracy: 97.02%, Time Passed: 6.61m
Epoch 82, Batch 300, Loss: 0.0863, Accuracy: 96.93%, Time Passed: 

In [63]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [64]:
del training_states

In [66]:
start_epoch = 84
end_epoch = 90
start_epoch, end_epoch

(84, 90)

In [67]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 84 - Training Started...

Epoch 84, Batch 100, Loss: 0.0790, Accuracy: 97.38%, Time Passed: 3.11m
Epoch 84, Batch 200, Loss: 0.0744, Accuracy: 97.38%, Time Passed: 6.26m
Epoch 84, Batch 300, Loss: 0.0765, Accuracy: 97.36%, Time Passed: 9.49m
Epoch 84, Batch 400, Loss: 0.0735, Accuracy: 97.39%, Time Passed: 12.85m
Epoch 84, Batch 500, Loss: 0.0759, Accuracy: 97.22%, Time Passed: 16.14m
Epoch 84 Completed - Average Loss: 0.0764, Accuracy: 97.25%, Epoch Time: 17.52m
Training Complete Epoch 84 - Total Time: 17.52m

✅ Epoch 84 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7651, Accuracy: 80.19%, Time Passed: 3.89m
Evaluation Complete - Loss: 0.6739, Accuracy: 83.27%, Total Time: 6.58m

📊 Epoch 84 - Evaluation Completed.


🚀 Epoch 85 - Training Started...

Epoch 85, Batch 100, Loss: 0.0799, Accuracy: 96.53%, Time Passed: 3.11m
Epoch 85, Batch 200, Loss: 0.0844, Accuracy: 96.77%, Time Passed: 6.28m
Epoch 85, Batch 300, Loss: 0.0831, Accuracy: 96.86%, Time Passed: 9

In [71]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [72]:
del training_states

In [74]:
start_epoch = 91
end_epoch = 95
start_epoch, end_epoch

(91, 95)

In [75]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 91 - Training Started...

Epoch 91, Batch 100, Loss: 0.0604, Accuracy: 97.66%, Time Passed: 3.36m
Epoch 91, Batch 200, Loss: 0.0697, Accuracy: 97.34%, Time Passed: 6.90m
Epoch 91, Batch 300, Loss: 0.0713, Accuracy: 97.24%, Time Passed: 10.42m
Epoch 91, Batch 400, Loss: 0.0702, Accuracy: 97.33%, Time Passed: 13.74m
Epoch 91, Batch 500, Loss: 0.0686, Accuracy: 97.38%, Time Passed: 17.10m
Epoch 91 Completed - Average Loss: 0.0678, Accuracy: 97.39%, Epoch Time: 18.50m
Training Complete Epoch 91 - Total Time: 18.50m

✅ Epoch 91 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7115, Accuracy: 80.94%, Time Passed: 4.01m
Evaluation Complete - Loss: 0.6577, Accuracy: 83.40%, Total Time: 6.82m

📊 Epoch 91 - Evaluation Completed.


🚀 Epoch 92 - Training Started...

Epoch 92, Batch 100, Loss: 0.0710, Accuracy: 97.59%, Time Passed: 3.31m
Epoch 92, Batch 200, Loss: 0.0731, Accuracy: 97.31%, Time Passed: 6.62m
Epoch 92, Batch 300, Loss: 0.0738, Accuracy: 97.34%, Time Passed: 

In [76]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [77]:
del training_states

In [79]:
start_epoch = 96
end_epoch = 100
start_epoch, end_epoch

(96, 100)

In [80]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageD, train_loader, val_loader, inverse_frequency_criterion, stageD_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageD',
                                                          save_dir = 'mobilenetv2', class_weights='inverse_frequency', log_interval=100)


🚀 Epoch 96 - Training Started...

Epoch 96, Batch 100, Loss: 0.0696, Accuracy: 97.31%, Time Passed: 3.24m
Epoch 96, Batch 200, Loss: 0.0639, Accuracy: 97.66%, Time Passed: 6.52m
Epoch 96, Batch 300, Loss: 0.0680, Accuracy: 97.39%, Time Passed: 9.85m
Epoch 96, Batch 400, Loss: 0.0684, Accuracy: 97.44%, Time Passed: 13.37m
Epoch 96, Batch 500, Loss: 0.0680, Accuracy: 97.47%, Time Passed: 16.81m
Epoch 96 Completed - Average Loss: 0.0671, Accuracy: 97.51%, Epoch Time: 18.22m
Training Complete Epoch 96 - Total Time: 18.22m

✅ Epoch 96 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7531, Accuracy: 79.97%, Time Passed: 3.95m
Evaluation Complete - Loss: 0.6785, Accuracy: 82.79%, Total Time: 6.73m

📊 Epoch 96 - Evaluation Completed.


🚀 Epoch 97 - Training Started...

Epoch 97, Batch 100, Loss: 0.0840, Accuracy: 97.12%, Time Passed: 3.40m
Epoch 97, Batch 200, Loss: 0.0753, Accuracy: 97.45%, Time Passed: 6.90m
Epoch 97, Batch 300, Loss: 0.0737, Accuracy: 97.46%, Time Passed: 1

In [81]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

## **Stage E**

### **98-110 & Amplified Inverse Recall + Inverse Frequency Combination**

In [56]:
df = pd.read_csv("mobilenetv2/mobilenetv2_stageD_val_results96_100.csv")
df = df[(df['Epoch'] == 97) & (df['Species'] != 'Average Performance (Macro)')]
recall_dict = dict(zip(df['Species'], df['Recall']))
inverse_recall_weights = compute_inverse_metric_tensor(recall_dict, train_dataset.class_to_idx, normalize=False, clip = False)

In [42]:
w = inverse_recall_weights.cpu()

# Compute percentiles
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
thresholds = [np.percentile(w.numpy(), p) for p in percentiles]
# Start with multipliers = 1
multipliers = torch.ones_like(w)

# Apply scaling in descending order to avoid overwriting smaller thresholds
rules = list(zip(thresholds, [1, 1, 2, 2, 3, 3, 4, 4, 5, 5]))
for t, m in rules:
    multipliers[w >= t] = m

# Apply the multipliers
amplified = w * multipliers
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
amplified = amplified.to(device)

In [43]:
amplified

tensor([ 2.0541,  5.7810,  4.9735,  2.0769,  4.8547,  2.1239,  3.3038,  1.0000,
         1.0000,  5.1810,  3.2727,  3.6213, 11.5625,  5.1600,  3.4459,  3.4915,
         1.0000,  5.5625,  7.6923,  3.4889,  2.0750,  1.0000,  1.0000,  1.0000,
         1.0000,  1.0000,  7.7295,  8.1068,  1.0087,  2.0964,  3.3837,  1.0000,
         2.0619], device='mps:0')

In [44]:
inverse_frequency_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=False)
# Normalize both to z-score
norm_inv_freq = (inverse_frequency_weights - inverse_frequency_weights.mean()) / inverse_frequency_weights.std()
norm_inv_recall = (amplified - amplified.mean()) / amplified.std()

# Combine and shift
combined_weights = norm_inv_freq + norm_inv_recall
combined_weights = combined_weights - combined_weights.min() + 1.0

# Optional: normalize mean to 1
combined_weights = combined_weights / combined_weights.mean()
combined_weights

tensor([2.0624, 2.6081, 2.2515, 2.2402, 2.5710, 1.8339, 3.1689, 2.9047, 3.0458,
        2.8874, 1.7250, 1.7781, 5.3387, 2.6111, 1.8430, 1.8588, 3.9966, 3.3798,
        3.2539, 1.8320, 1.0000, 5.0856, 1.4917, 1.2945, 1.3354, 1.2498, 3.2678,
        4.2951, 1.1903, 1.1804, 1.8226, 2.3101, 1.7214], device='mps:0')

In [153]:
combined_criterion = nn.CrossEntropyLoss(weight=combined_weights)   

In [161]:
training_states = joblib.load(f'mobilenetv2/training_states_{96}_{100}.pkl')

In [162]:
mobilenetv2_stageE = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageE.to(device)
mobilenetv2_stageE.load_state_dict(training_states[97]["model_state_dict"])

<All keys matched successfully>

In [163]:
mobilenetv2_stageE = configure_mobilenetv2_training(
                                                    mobilenetv2_stageE,
                                                    NUM_CLASSES, final_stage=True,
                                                    prev_unfrozen = 0
                                                   )
    
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageE)
df_summary

🟢 Final stage → Entire backbone unfrozen (blocks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
✅ Trainable layers: 158/158 (100.00%)
✅ Trainable params: 2,266,145/2,266,145 (100.00%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,928,1.898734,0.040951,0.040951,158,100.000000,2266145,100.000000,1.923077,0.041729,0.041729,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,896,3.797468,0.039539,0.039539,155,98.101266,2265217,99.959049,3.846154,0.040290,0.040290,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,5136,5.696203,0.226640,0.226640,149,94.303797,2264321,99.919511,5.769231,0.230949,0.230949,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,8832,5.696203,0.389737,0.389737,140,88.607595,2259185,99.692870,5.769231,0.397145,0.397145,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,10000,5.696203,0.441278,0.441278,131,82.911392,2250353,99.303134,5.769231,0.449666,0.449666,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,14848,5.696203,0.655210,0.655210,122,77.215190,2240353,98.861856,5.769231,0.667664,0.667664,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,14848,5.696203,0.655210,0.655210,113,71.518987,2225505,98.206646,5.769231,0.667664,0.667664,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,21056,5.696203,0.929155,0.929155,104,65.822785,2210657,97.551436,5.769231,0.946817,0.946817,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,54272,5.696203,2.394904,2.394904,95,60.126582,2189601,96.622281,5.769231,2.440428,2.440428,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,54272,5.696203,2.394904,2.394904,86,54.430380,2135329,94.227377,5.769231,2.440428,2.440428,84,53.846154,2093056,94.117647


In [95]:
_ = evaluate_model(
            mobilenetv2_stageE, val_loader, inverse_frequency_criterion, train_dataset.classes, epoch_num=97, log_interval=100)

Batch 100, Loss: 0.7312, Accuracy: 81.28%, Time Passed: 3.65m
Evaluation Complete - Loss: 0.6413, Accuracy: 84.06%, Total Time: 6.37m


In [164]:
stageE_optimizer = torch.optim.AdamW(
    mobilenetv2_stageE.parameters(),
    lr=1e-5,    # even smaller LR, since more params unfrozen
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

In [165]:
del training_states

In [166]:
start_epoch = 98
end_epoch = 102
start_epoch, end_epoch

(98, 102)

In [167]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='combined', log_interval=100)


🚀 Epoch 98 - Training Started...

Epoch 98, Batch 100, Loss: 0.0721, Accuracy: 97.78%, Time Passed: 3.54m
Epoch 98, Batch 200, Loss: 0.0686, Accuracy: 97.75%, Time Passed: 7.45m
Epoch 98, Batch 300, Loss: 0.0671, Accuracy: 97.81%, Time Passed: 11.11m
Epoch 98, Batch 400, Loss: 0.0689, Accuracy: 97.76%, Time Passed: 14.90m
Epoch 98, Batch 500, Loss: 0.0693, Accuracy: 97.74%, Time Passed: 18.62m
Epoch 98 Completed - Average Loss: 0.0699, Accuracy: 97.71%, Epoch Time: 20.25m
Training Complete Epoch 98 - Total Time: 20.25m

✅ Epoch 98 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.7160, Accuracy: 80.97%, Time Passed: 4.03m
Evaluation Complete - Loss: 0.6329, Accuracy: 83.79%, Total Time: 6.87m

📊 Epoch 98 - Evaluation Completed.


🚀 Epoch 99 - Training Started...

Epoch 99, Batch 100, Loss: 0.0615, Accuracy: 97.53%, Time Passed: 4.25m
Epoch 99, Batch 200, Loss: 0.0611, Accuracy: 97.73%, Time Passed: 8.47m
Epoch 99, Batch 300, Loss: 0.0597, Accuracy: 97.79%, Time Passed: 

In [169]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [170]:
del training_states

In [173]:
start_epoch = 103
end_epoch = 110
start_epoch, end_epoch

(103, 110)

In [174]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='combined', log_interval=100)


🚀 Epoch 103 - Training Started...

Epoch 103, Batch 100, Loss: 0.0612, Accuracy: 97.91%, Time Passed: 4.06m
Epoch 103, Batch 200, Loss: 0.0624, Accuracy: 97.92%, Time Passed: 8.58m
Epoch 103, Batch 300, Loss: 0.0631, Accuracy: 97.88%, Time Passed: 13.02m
Epoch 103, Batch 400, Loss: 0.0626, Accuracy: 97.88%, Time Passed: 17.40m
Epoch 103, Batch 500, Loss: 0.0615, Accuracy: 97.91%, Time Passed: 21.78m
Epoch 103 Completed - Average Loss: 0.0613, Accuracy: 97.89%, Epoch Time: 24.01m
Training Complete Epoch 103 - Total Time: 24.01m

✅ Epoch 103 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.6675, Accuracy: 82.22%, Time Passed: 4.83m
Evaluation Complete - Loss: 0.6333, Accuracy: 84.35%, Total Time: 7.83m

📊 Epoch 103 - Evaluation Completed.


🚀 Epoch 104 - Training Started...

Epoch 104, Batch 100, Loss: 0.0616, Accuracy: 97.69%, Time Passed: 4.54m
Epoch 104, Batch 200, Loss: 0.0559, Accuracy: 98.03%, Time Passed: 9.04m
Epoch 104, Batch 300, Loss: 0.0545, Accuracy: 98.11%,

In [176]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [177]:
del training_states

### **109-120 & Moderately Recall-Weighted Combination (Amplified Inverse Recall + Inverse Frequency, 2.5 : 1/2.5 Ratio)**

In [266]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


In [267]:
df = pd.read_csv("mobilenetv2/mobilenetv2_stageE_val_results103_110.csv")
df = df[(df['Epoch'] == 108) & (df['Species'] != 'Average Performance (Macro)')]

In [268]:
# Compute from recall values
recall_penalty_weights = (1 - df['Recall'].values) * 100

# Convert to tensor
recall_penalty_weights = torch.tensor(recall_penalty_weights, dtype=torch.float32)

# Move to MPS (Apple GPU) if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
recall_penalty_weights = recall_penalty_weights.to(device)

In [269]:
recall_penalty_weights

tensor([ 2.6316, 23.7374,  9.9644,  4.9383, 18.3099, 13.3333,  8.0460,  0.0000,
         0.0000, 20.5882,  3.4314, 13.2353, 45.9459, 12.4031, 10.5882, 14.0777,
         0.0000, 24.7191, 33.5714, 18.7898,  4.2169,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000, 37.8125, 42.5150,  0.8621,  4.2146, 10.3093,  0.8772,
         3.0000], device='mps:0')

In [270]:
inverse_frequency_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=False)
inverse_frequency_criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights)
inverse_frequency_weights

tensor([ 58.0736,  20.0740,  18.1442,  65.2782,  33.7165,  47.1848,  84.7024,
        111.3077, 117.3243,  41.8410,  23.6567,  20.1907,  41.4415,  30.4098,
         25.8393,  25.7626, 157.8545,  56.5603,  16.1827,  24.6648,  12.4384,
        204.2823,  51.0706,  42.6634,  44.4092,  40.7606,  16.1676,  53.7585,
         38.0789,  19.7768,  25.9940,  85.9604,  43.4100], device='mps:0')

In [271]:
# Normalize both to z-score
norm_inv_freq = (inverse_frequency_weights - inverse_frequency_weights.mean()) / inverse_frequency_weights.std()
norm_recall_penalty = (recall_penalty_weights - recall_penalty_weights.mean()) / recall_penalty_weights.std()

# Combine and shift
new_combined_weights = (norm_inv_freq/2.5) + (2.5*norm_recall_penalty)
new_combined_weights = new_combined_weights - new_combined_weights.min() + 1.0
new_combined_weights

tensor([1.6636, 5.3267, 2.6855, 2.1706, 4.4211, 3.5996, 2.9447, 1.6619, 1.7184,
        4.9312, 1.4930, 3.3277, 9.7569, 3.2651, 2.8765, 3.5404, 2.0987, 5.8560,
        7.1631, 4.4275, 1.5374, 2.5343, 1.0967, 1.0179, 1.0342, 1.0000, 7.9707,
        9.2190, 1.1390, 1.6058, 2.8249, 1.5912, 1.5962], device='mps:0')

In [272]:
new_combined_criterion = nn.CrossEntropyLoss(weight=new_combined_weights)   

In [275]:
training_states = joblib.load(f'mobilenetv2/training_states_{103}_{110}.pkl')

In [276]:
mobilenetv2_stageE = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageE.to(device)
mobilenetv2_stageE.load_state_dict(training_states[108]["model_state_dict"])

<All keys matched successfully>

In [277]:
mobilenetv2_stageE = configure_mobilenetv2_training(
                                                    mobilenetv2_stageE,
                                                    NUM_CLASSES, final_stage=True,
                                                    prev_unfrozen = 0
                                                   )
    
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageE)
df_summary

🟢 Final stage → Entire backbone unfrozen (blocks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
✅ Trainable layers: 158/158 (100.00%)
✅ Trainable params: 2,266,145/2,266,145 (100.00%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,928,1.898734,0.040951,0.040951,158,100.000000,2266145,100.000000,1.923077,0.041729,0.041729,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,896,3.797468,0.039539,0.039539,155,98.101266,2265217,99.959049,3.846154,0.040290,0.040290,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,5136,5.696203,0.226640,0.226640,149,94.303797,2264321,99.919511,5.769231,0.230949,0.230949,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,8832,5.696203,0.389737,0.389737,140,88.607595,2259185,99.692870,5.769231,0.397145,0.397145,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,10000,5.696203,0.441278,0.441278,131,82.911392,2250353,99.303134,5.769231,0.449666,0.449666,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,14848,5.696203,0.655210,0.655210,122,77.215190,2240353,98.861856,5.769231,0.667664,0.667664,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,14848,5.696203,0.655210,0.655210,113,71.518987,2225505,98.206646,5.769231,0.667664,0.667664,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,21056,5.696203,0.929155,0.929155,104,65.822785,2210657,97.551436,5.769231,0.946817,0.946817,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,54272,5.696203,2.394904,2.394904,95,60.126582,2189601,96.622281,5.769231,2.440428,2.440428,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,54272,5.696203,2.394904,2.394904,86,54.430380,2135329,94.227377,5.769231,2.440428,2.440428,84,53.846154,2093056,94.117647


In [279]:
_ = evaluate_model(
            mobilenetv2_stageE, val_loader, new_combined_criterion, train_dataset.classes, epoch_num=108, log_interval=100)

Batch 100, Loss: 0.6301, Accuracy: 83.75%, Time Passed: 3.89m
Evaluation Complete - Loss: 0.6058, Accuracy: 85.31%, Total Time: 6.69m


In [280]:
stageE_optimizer = torch.optim.AdamW(
    mobilenetv2_stageE.parameters(),
    lr=1e-5,    # even smaller LR, since more params unfrozen
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

In [281]:
start_epoch = 109
end_epoch = 115
start_epoch, end_epoch

(109, 115)

In [282]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, new_combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='new_combined', log_interval=100)


🚀 Epoch 109 - Training Started...

Epoch 109, Batch 100, Loss: 0.0494, Accuracy: 98.06%, Time Passed: 3.87m
Epoch 109, Batch 200, Loss: 0.0522, Accuracy: 98.03%, Time Passed: 7.65m
Epoch 109, Batch 300, Loss: 0.0502, Accuracy: 98.14%, Time Passed: 11.50m
Epoch 109, Batch 400, Loss: 0.0529, Accuracy: 98.04%, Time Passed: 15.32m
Epoch 109, Batch 500, Loss: 0.0544, Accuracy: 98.02%, Time Passed: 19.12m
Epoch 109 Completed - Average Loss: 0.0548, Accuracy: 98.05%, Epoch Time: 20.64m
Training Complete Epoch 109 - Total Time: 20.64m

✅ Epoch 109 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5957, Accuracy: 84.91%, Time Passed: 4.18m
Evaluation Complete - Loss: 0.5727, Accuracy: 86.12%, Total Time: 7.11m

📊 Epoch 109 - Evaluation Completed.


🚀 Epoch 110 - Training Started...

Epoch 110, Batch 100, Loss: 0.0491, Accuracy: 98.22%, Time Passed: 3.96m
Epoch 110, Batch 200, Loss: 0.0447, Accuracy: 98.28%, Time Passed: 7.91m
Epoch 110, Batch 300, Loss: 0.0522, Accuracy: 98.14%,

In [283]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [284]:
del training_states

In [285]:
start_epoch = 116
end_epoch = 118
start_epoch, end_epoch

(116, 118)

In [286]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, new_combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='new_combined', log_interval=100)


🚀 Epoch 116 - Training Started...

Epoch 116, Batch 100, Loss: 0.0390, Accuracy: 98.59%, Time Passed: 3.82m
Epoch 116, Batch 200, Loss: 0.0428, Accuracy: 98.50%, Time Passed: 7.49m
Epoch 116, Batch 300, Loss: 0.0443, Accuracy: 98.38%, Time Passed: 11.22m
Epoch 116, Batch 400, Loss: 0.0428, Accuracy: 98.30%, Time Passed: 15.00m
Epoch 116, Batch 500, Loss: 0.0441, Accuracy: 98.23%, Time Passed: 18.74m
Epoch 116 Completed - Average Loss: 0.0448, Accuracy: 98.22%, Epoch Time: 20.27m
Training Complete Epoch 116 - Total Time: 20.27m

✅ Epoch 116 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5864, Accuracy: 84.09%, Time Passed: 4.13m
Evaluation Complete - Loss: 0.5572, Accuracy: 85.92%, Total Time: 6.98m

📊 Epoch 116 - Evaluation Completed.


🚀 Epoch 117 - Training Started...

Epoch 117, Batch 100, Loss: 0.0432, Accuracy: 98.31%, Time Passed: 3.93m
Epoch 117, Batch 200, Loss: 0.0412, Accuracy: 98.41%, Time Passed: 7.97m
Epoch 117, Batch 300, Loss: 0.0419, Accuracy: 98.44%,

In [287]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')

In [288]:
del training_states

In [290]:
start_epoch = 119
end_epoch = 120
start_epoch, end_epoch

(119, 120)

In [291]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, new_combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='new_combined', log_interval=100)


🚀 Epoch 119 - Training Started...

Epoch 119, Batch 100, Loss: 0.0315, Accuracy: 98.75%, Time Passed: 3.71m
Epoch 119, Batch 200, Loss: 0.0360, Accuracy: 98.55%, Time Passed: 7.61m
Epoch 119, Batch 300, Loss: 0.0370, Accuracy: 98.46%, Time Passed: 11.45m
Epoch 119, Batch 400, Loss: 0.0389, Accuracy: 98.44%, Time Passed: 15.33m
Epoch 119, Batch 500, Loss: 0.0392, Accuracy: 98.39%, Time Passed: 19.43m
Epoch 119 Completed - Average Loss: 0.0391, Accuracy: 98.41%, Epoch Time: 21.10m
Training Complete Epoch 119 - Total Time: 21.10m

✅ Epoch 119 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.5764, Accuracy: 84.97%, Time Passed: 4.33m
Evaluation Complete - Loss: 0.5543, Accuracy: 86.37%, Total Time: 7.30m

📊 Epoch 119 - Evaluation Completed.


🚀 Epoch 120 - Training Started...

Epoch 120, Batch 100, Loss: 0.0328, Accuracy: 98.50%, Time Passed: 4.09m
Epoch 120, Batch 200, Loss: 0.0338, Accuracy: 98.50%, Time Passed: 8.43m
Epoch 120, Batch 300, Loss: 0.0368, Accuracy: 98.41%,

In [ ]:
if training_states:
    joblib.dump(
        training_states,
        'mobilenetv2/training_states_119_120.pkl'
    )

### **121-130 & Strong Recall-Weighted Combination (Amplified Inverse Recall + Inverse Frequency, 5 : 1/5 Ratio)**

In [108]:
state, generator = setup_reproducibility(start_fresh=True, seed=seeds[0])

✅ Random seed set to 42 (MPS available: True)


In [102]:
df = pd.read_csv("mobilenetv2/mobilenetv2_stageE_val_results119_120.csv")
df = df[(df['Epoch'] == 120) & (df['Species'] != 'Average Performance (Macro)')]

In [103]:
# Compute from recall values
recall_penalty_weights = (1 - df['Recall'].values) * 100

# Convert to tensor
recall_penalty_weights = torch.tensor(recall_penalty_weights, dtype=torch.float32)

# Move to MPS (Apple GPU) if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
recall_penalty_weights = recall_penalty_weights.to(device)

In [104]:
recall_penalty_weights

tensor([ 2.6316, 20.7071, 11.7438,  4.9383, 21.1268, 12.5000,  5.7471,  0.0000,
         0.0000, 19.1176,  6.3725, 10.7843, 46.4865, 10.8527, 10.5882, 11.6505,
         0.0000, 25.8427, 27.8571, 20.0637,  3.8153,  0.0000,  0.0000,  0.9524,
         0.0000,  0.0000, 35.0000, 43.1138,  1.7241,  3.0651,  7.7320,  1.7544,
         3.0000], device='mps:0')

In [105]:
inverse_frequency_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=False)
inverse_frequency_criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights)
inverse_frequency_weights

tensor([ 58.0736,  20.0740,  18.1442,  65.2782,  33.7165,  47.1848,  84.7024,
        111.3077, 117.3243,  41.8410,  23.6567,  20.1907,  41.4415,  30.4098,
         25.8393,  25.7626, 157.8545,  56.5603,  16.1827,  24.6648,  12.4384,
        204.2823,  51.0706,  42.6634,  44.4092,  40.7606,  16.1676,  53.7585,
         38.0789,  19.7768,  25.9940,  85.9604,  43.4100], device='mps:0')

In [106]:
# Normalize both to z-score
norm_inv_freq = (inverse_frequency_weights - inverse_frequency_weights.mean()) / inverse_frequency_weights.std()
norm_recall_penalty = (recall_penalty_weights - recall_penalty_weights.mean()) / recall_penalty_weights.std()

# Combine and shift
new_combined_weights = (norm_inv_freq/5) + (5*norm_recall_penalty)
new_combined_weights = new_combined_weights - new_combined_weights.min() + 1.0
new_combined_weights

tensor([ 2.1151,  9.0385,  5.5079,  3.0552,  9.2674,  5.9412,  3.4641,  1.3310,
         1.3592,  8.5161,  3.4234,  5.1405, 19.2671,  5.2153,  5.0900,  5.5069,
         1.5493, 11.2274, 11.8294,  8.8072,  2.3661,  1.7672,  1.0484,  1.3831,
         1.0171,  1.0000, 14.6356, 17.9998,  1.6648,  2.1058,  3.9685,  1.9013,
         2.1911], device='mps:0')

In [107]:
new_combined_criterion = nn.CrossEntropyLoss(weight=new_combined_weights)   

In [118]:
training_states = joblib.load(f'mobilenetv2/training_states_{119}_{120}.pkl')

In [119]:
mobilenetv2_stageE = models.mobilenet_v2(weights=None, num_classes=len(train_dataset.classes))

# Send it to device
device = torch.device("mps")
mobilenetv2_stageE.to(device)
mobilenetv2_stageE.load_state_dict(training_states[120]["model_state_dict"])

<All keys matched successfully>

In [120]:
mobilenetv2_stageE = configure_mobilenetv2_training(
                                                    mobilenetv2_stageE,
                                                    NUM_CLASSES, final_stage=True,
                                                    prev_unfrozen = 0
                                                   )
    
df_summary = summarize_mobilenetv2_blocks(mobilenetv2_stageE)
df_summary

🟢 Final stage → Entire backbone unfrozen (blocks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
✅ Trainable layers: 158/158 (100.00%)
✅ Trainable params: 2,266,145/2,266,145 (100.00%)


,Block,Order,Module,Layers (#),Params (#),Trainable Params (#),% Layers (Model),% Params (Model),% Trainable Params (Model),Cumulative Layers (Model),Cumulative % Layers (Model),Cumulative Params (Model),Cumulative % Params (Model),Layers (Module %),Params (Module %),Trainable Params (Module %),Cumulative Layers (Module),Cumulative % Layers (Module),Cumulative Params (Module),Cumulative % Params (Module)
0,Conv2dNormActivation_1,1,features,3,928,928,1.898734,0.040951,0.040951,158,100.000000,2266145,100.000000,1.923077,0.041729,0.041729,156,100.000000,2223872,100.000000
1,InvertedResidual_2,2,features,6,896,896,3.797468,0.039539,0.039539,155,98.101266,2265217,99.959049,3.846154,0.040290,0.040290,153,98.076923,2222944,99.958271
2,InvertedResidual_3,3,features,9,5136,5136,5.696203,0.226640,0.226640,149,94.303797,2264321,99.919511,5.769231,0.230949,0.230949,147,94.230769,2222048,99.917981
3,InvertedResidual_4,4,features,9,8832,8832,5.696203,0.389737,0.389737,140,88.607595,2259185,99.692870,5.769231,0.397145,0.397145,138,88.461538,2216912,99.687032
4,InvertedResidual_5,5,features,9,10000,10000,5.696203,0.441278,0.441278,131,82.911392,2250353,99.303134,5.769231,0.449666,0.449666,129,82.692308,2208080,99.289887
5,InvertedResidual_6,6,features,9,14848,14848,5.696203,0.655210,0.655210,122,77.215190,2240353,98.861856,5.769231,0.667664,0.667664,120,76.923077,2198080,98.840221
6,InvertedResidual_7,7,features,9,14848,14848,5.696203,0.655210,0.655210,113,71.518987,2225505,98.206646,5.769231,0.667664,0.667664,111,71.153846,2183232,98.172557
7,InvertedResidual_8,8,features,9,21056,21056,5.696203,0.929155,0.929155,104,65.822785,2210657,97.551436,5.769231,0.946817,0.946817,102,65.384615,2168384,97.504892
8,InvertedResidual_9,9,features,9,54272,54272,5.696203,2.394904,2.394904,95,60.126582,2189601,96.622281,5.769231,2.440428,2.440428,93,59.615385,2147328,96.558075
9,InvertedResidual_10,10,features,9,54272,54272,5.696203,2.394904,2.394904,86,54.430380,2135329,94.227377,5.769231,2.440428,2.440428,84,53.846154,2093056,94.117647


In [121]:
stageE_optimizer = torch.optim.AdamW(
    mobilenetv2_stageE.parameters(),
    lr=1e-5,    # even smaller LR, since more params unfrozen
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

In [122]:
start_epoch = 121
end_epoch = 130
start_epoch, end_epoch

(121, 130)

In [123]:
training_states, training_logs = train_and_evaluate_model(mobilenetv2_stageE, train_loader, val_loader, new_combined_criterion, stageE_optimizer,
                                                          train_dataset.classes, start_epoch, end_epoch, model_name = 'mobilenetv2_stageE',
                                                          save_dir = 'mobilenetv2', class_weights='new_combined', log_interval=100)


🚀 Epoch 121 - Training Started...

Epoch 121, Batch 100, Loss: 0.0362, Accuracy: 98.41%, Time Passed: 4.29m
Epoch 121, Batch 200, Loss: 0.0354, Accuracy: 98.55%, Time Passed: 8.33m
Epoch 121, Batch 300, Loss: 0.0350, Accuracy: 98.50%, Time Passed: 12.34m
Epoch 121, Batch 400, Loss: 0.0359, Accuracy: 98.45%, Time Passed: 16.53m
Epoch 121, Batch 500, Loss: 0.0378, Accuracy: 98.41%, Time Passed: 20.88m
Epoch 121 Completed - Average Loss: 0.0378, Accuracy: 98.41%, Epoch Time: 22.72m
Training Complete Epoch 121 - Total Time: 22.72m

✅ Epoch 121 - Training Completed. Starting Evaluation...

Batch 100, Loss: 0.6025, Accuracy: 84.62%, Time Passed: 4.66m
Evaluation Complete - Loss: 0.5854, Accuracy: 85.92%, Total Time: 7.75m

📊 Epoch 121 - Evaluation Completed.


🚀 Epoch 122 - Training Started...

Epoch 122, Batch 100, Loss: 0.0394, Accuracy: 98.34%, Time Passed: 4.26m
Epoch 122, Batch 200, Loss: 0.0411, Accuracy: 98.38%, Time Passed: 8.62m
Epoch 122, Batch 300, Loss: 0.0390, Accuracy: 98.32%,

In [124]:
if training_states:
    joblib.dump(training_states, f'mobilenetv2/training_states_{start_epoch}_{end_epoch}.pkl')